# ADAPT-BIO: Anticipatory Dynamic Attention Pruning Topology
### Biologically-Inspired Sparse Attention via Slime Mold Anticipation, RNA-Style Mask Re-Editing, and Starling Flock Topology

**Author:** Kritika Benjwal  
**Notebook:** 1 of 2 — BERT Tokenizer Experiments (WikiText-2 & WikiText-103)  
**Status:** Consolidated research notebook — WikiText-2 / WikiText-103 experiments, ablations, and scaling analysis

---

## Overview

ADAPT-BIO introduces **SOMA** (Slime mold / Octopus RNA / starling-Murmuration Attention), a sparse attention mechanism for transformers that fuses three biologically-inspired signals:

1. **Movement Pruning (Slime Mold Anticipation)** — tracks attention weight movement during an initial anticipation window to discover an importance signal before any pruning begins.
2. **RNA-Style Mask Refinement** — periodically re-edits the sparsity mask every Δ steps based on the accumulated movement signal (this notebook uses the **corrected, continuously-accumulating** version of this signal — see Section 2).
3. **Starling Topology Constraint** — enforces a fixed top-k=7 neighbor sparsity pattern per query, inspired by local-interaction rules in starling murmurations.

## What this notebook contains

| Section | Contents |
|---|---|
| 1 | Environment setup & configuration |
| 2 | SOMA core components (with corrected dynamic movement signal) |
| 3 | ADAPT-BIO transformer architecture |
| 4 | Data loading — WikiText-2 and WikiText-103 |
| 5 | Training & benchmarking utilities (time, memory, FLOPs) |
| 6 | Main results — Dense vs ADAPT-BIO at d_model=128 and d_model=256 (WikiText-2) |
| 7 | k-sweep ablation (WikiText-2) |
| 8 | Δ (update interval) sweep (WikiText-2) |
| 9 | Component ablation — Full / No-RNA / No-Starling (WikiText-2) |
| 10 | T=512 sequence length scaling experiment (WikiText-2) |
| 11 | Real wall-clock time & GPU memory benchmarks (WikiText-2) |
| 12 | Aggregated results, tables, and figures (WikiText-2) |
| 14 | WikiText-103 experiments — all sections repeated |
| 15–21 | WikiText-103 main results, ablations, scaling, benchmarks, summary |

> **Note:** All results are saved incrementally to `/kaggle/working/results.json` after each run, so the notebook is resumable and individual experiment blocks can be re-run independently without redoing everything.  
> **Notebook 2** (GPT2 tokenizer experiments) runs independently and produces its own `results.json`.

## 1. Environment Setup

This cell clones the ADAPT-BIO repository and installs dependencies.

> ⚠️ **Security note:** Your GitHub token must come from **Kaggle Secrets**, not be hardcoded in the notebook. Before running this cell for the first time:
> 1. Revoke the old exposed token at https://github.com/settings/tokens (if you haven't already)
> 2. Generate a new fine-scoped token
> 3. In this notebook: **Add-ons → Secrets → Add Secret**, name it `GITHUB_TOKEN`, paste the new token
>
> The cell below reads it via `UserSecretsClient` — it will never appear in plaintext, in version history, or in any exported `.ipynb`.

In [1]:
!git config --global user.name "Kritika Benjwal"
!git config --global user.email "ananya.benjwal@gmail.com"

In [2]:
import os, sys, time

# ── GitHub token from Kaggle Secrets (never hardcoded) ──────────────────────
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")

!git clone https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git
%cd /kaggle/working/adapt-bio
!pip install -q einops omegaconf datasets transformers

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("PyTorch:", torch.__version__)

# Make local repo importable
sys.path.insert(0, '/kaggle/working/adapt-bio/src')

# Results store — every later section appends here and saves after each run
import json
RESULTS_PATH = '/kaggle/working/results.json'
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as f:
        results = json.load(f)
    print(f"Loaded existing results.json with keys: {list(results.keys())}")
else:
    results = {}
    print("Starting fresh results.json")

def save_results():
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f, indent=2, default=str)
    print(f"✅ results.json saved ({list(results.keys())})")

Cloning into 'adapt-bio'...
remote: Enumerating objects: 562, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 562 (delta 65), reused 149 (delta 36), pack-reused 381 (from 1)
Receiving objects: 100% (562/562), 7.89 MiB | 39.24 MiB/s, done.
Resolving deltas: 100% (224/224), done.
/kaggle/working/adapt-bio
CUDA available: True
Device: Tesla T4
PyTorch: 2.10.0+cu128
Starting fresh results.json


## 2. SOMA Core Components

All three bio-inspired modules are defined inline here (no `src/` import needed for standalone runs).  
Key fix vs earlier notebooks: `SOMAMask.forward` stays **dense** during the anticipation window (`step < anticipation_steps`) and only starts emitting a sparse mask after the window closes.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Signal 3 (listed first because Signal 1 depends on it) ──────────────────

class StarlingTopologyConstraint(nn.Module):
    """
    Starling murmuration topology: each token attends to exactly its k
    most movement-active neighbours (k=7 by default, per Ballerini et al. 2008).
    """
    def __init__(self, k: int = 7):
        super().__init__()
        self.k = k

    def apply(self, movement_scores: torch.Tensor) -> torch.Tensor:
        """
        Args:
            movement_scores: (H, T, T)  – accumulated movement signal
        Returns:
            boolean mask:   (H, T, T)  – True for the top-k positions per row
        """
        k = min(self.k, movement_scores.shape[-1])
        _, topk_idx = movement_scores.topk(k, dim=-1)          # (H, T, k)
        mask = torch.zeros_like(movement_scores, dtype=torch.bool)
        mask.scatter_(-1, topk_idx, True)
        return mask


# ── Signal 1 ────────────────────────────────────────────────────────────────

class MovementPruningMask(nn.Module):
    """
    Slime-mould anticipation: accumulates step-wise attention-weight movement
    to discover which edges are gradient-sensitive.

    Two-phase accumulation (journal version — fixes Eq.1 in conference paper):

      Anticipation window (step ≤ S):
        A_h^(t) = Σ_{τ=1}^{t} |W_h^(τ) − W_h^(τ-1)|          (cumulative, Eq.1)

      Post-anticipation (step > S):
        A_h^(t) = (1 − α) · A_h^(t-1) + α · |W_h^(t) − W_h^(t-1)|   (EMA, Eq.1b)

    The EMA phase is the key journal addition over the conference paper:
    it keeps the movement signal live and responsive to the evolving gradient
    landscape, making each RNA re-edit a genuine revision rather than a
    deterministic replay of a frozen 100-step snapshot.
    α (ema_alpha) is a new hyperparameter; 0.05 was found optimal via
    informal sweep and is fixed for all reported experiments.
    """

    def __init__(self,
                 num_heads: int,
                 seq_len: int,
                 anticipation_steps: int,
                 ema_alpha: float = 0.05):
        super().__init__()
        self.anticipation_steps = anticipation_steps
        self.ema_alpha = ema_alpha
        self.step = 0
        self.register_buffer("movement_accum",
                             torch.zeros(num_heads, seq_len, seq_len))
        self.register_buffer("prev_weights",
                             torch.zeros(num_heads, seq_len, seq_len))

    def update(self, attn_weights: torch.Tensor) -> None:
        """
        Called once per forward pass with the attention signal
        averaged over the batch: shape (H, T, T).
        """
        w = attn_weights.detach()

        if self.step > 0:
            delta = (w - self.prev_weights).abs()

            if self.step <= self.anticipation_steps:
                # Cumulative sum — Eq. 1 (conference paper, also Eq. 1 journal)
                self.movement_accum.add_(delta)
            else:
                # EMA — Eq. 1b (journal addition)
                # Keeps signal live so RNA re-edits reflect current training state
                self.movement_accum.mul_(1.0 - self.ema_alpha).add_(
                    delta * self.ema_alpha
                )

        self.prev_weights.copy_(w)
        self.step += 1


# ── Signal 2 ────────────────────────────────────────────────────────────────

class RNAMaskRefinement(nn.Module):
    """
    Octopus-RNA self-editing: re-evaluates the sparsity mask every
    `update_interval` steps using the current (EMA-fresh) movement signal.
    """
    def __init__(self, update_interval: int = 500):
        super().__init__()
        self.update_interval  = update_interval
        self.last_update_step = 0

    def should_update(self, step: int) -> bool:
        return (step - self.last_update_step) >= self.update_interval

    def refine(self, current_mask, movement_signal, step, k=7):
        if not self.should_update(step):
            return current_mask
        new_mask = StarlingTopologyConstraint(k=k).apply(movement_signal)
        self.last_update_step = step
        return new_mask


# ── SOMA integration ─────────────────────────────────────────────────────────

class SOMAMask(nn.Module):
    """
    Self-Organising Mask Anticipation — combines all three biological signals.

    Timeline
    ────────
    steps 0 … anticipation_steps−1   : DENSE   (slime-mould sensing, cumulative Eq.1)
    steps anticipation_steps … Δ−1   : DENSE   (waiting for first RNA edit, EMA Eq.1b)
    step  Δ, 2Δ, 3Δ …               : SPARSE  (RNA re-edits with EMA-fresh signal)

    Ablation flags (Section 9):
    ┌──────────────────┬──────────┬────────────┬──────────────────────────────────┐
    │ Config           │ use_rna  │use_starling│ Behaviour                        │
    ├──────────────────┼──────────┼────────────┼──────────────────────────────────┤
    │ Full ADAPT-BIO   │ True     │ True       │ RNA + k=7 topology               │
    │ No-RNA           │ False    │ True       │ freeze once at S, k=7 topology   │
    │ No-Starling      │ True     │ False      │ RNA + k=T (dense topology)       │
    │ Movement-only    │ False    │ False      │ freeze once at S, raw top-k,     │
    │                  │          │            │ no bio topology or revision       │
    └──────────────────┴──────────┴────────────┴──────────────────────────────────┘
    """
    def __init__(self, num_heads, seq_len, k=7,
                 anticipation_steps=100, rna_update_interval=500,
                 ema_alpha=0.05,          # ← NEW: surfaced for ablation/tuning
                 use_rna=True, use_starling=True):
        super().__init__()
        self.anticipation_steps = anticipation_steps
        self.use_rna            = use_rna
        self.use_starling       = use_starling
        self.k                  = k

        self.movement = MovementPruningMask(
            num_heads, seq_len, anticipation_steps, ema_alpha=ema_alpha
        )
        self.rna = (RNAMaskRefinement(update_interval=rna_update_interval)
                    if use_rna else None)
        self.starling = StarlingTopologyConstraint(k=k) if use_starling else None

        self.register_buffer("current_mask",
                             torch.ones(num_heads, seq_len, seq_len,
                                        dtype=torch.bool))

    def forward(self, attn_weights: torch.Tensor, step: int) -> torch.Tensor:
        self.movement.update(attn_weights)

        if step < self.anticipation_steps:
            return self.current_mask

        signal = self.movement.movement_accum   # (H, T, T) — EMA-fresh post-S

        if self.use_rna:
            k_eff = self.k if self.use_starling else signal.shape[-1]
            self.current_mask = self.rna.refine(
                self.current_mask, signal, step=step, k=k_eff
            )
        elif self.use_starling and step == self.anticipation_steps:
            # No-RNA: freeze mask once at end of anticipation window
            self.current_mask = self.starling.apply(signal)
        elif not self.use_rna and not self.use_starling:
            # movement_only: freeze once using raw top-k, no bio topology label
            if step == self.anticipation_steps:
                self.current_mask = StarlingTopologyConstraint(k=self.k).apply(signal)

        return self.current_mask


# ── Smoke test ───────────────────────────────────────────────────────────────

def _smoke_test():
    print("Running architecture smoke tests ...\n")

    HEADS, T, ANTICI, K = 4, 32, 10, 7

    # Fix 1: EMA keeps movement_accum live after anticipation
    mm = MovementPruningMask(num_heads=HEADS, seq_len=T,
                             anticipation_steps=ANTICI, ema_alpha=0.05)
    snapshots = []
    for s in range(60):
        mm.update(torch.rand(HEADS, T, T))
        if s in (ANTICI - 1, ANTICI + 10, ANTICI + 20, ANTICI + 30):
            snapshots.append(mm.movement_accum.clone())

    same_after = all(torch.allclose(snapshots[0], sn) for sn in snapshots[1:])
    assert not same_after, "FAIL: movement_accum frozen after anticipation — EMA not applied."
    print("  Fix 1 ✅  movement_accum evolves after anticipation window (EMA active)\n")

    # Fix 2: StarlingTopologyConstraint gives exactly k per row
    sc   = StarlingTopologyConstraint(k=K)
    fake = torch.rand(HEADS, T, T)
    fake[0, 0, :K] = 0.99
    mask = sc.apply(fake)
    all_exactly_k = (mask.long().sum(dim=-1) == K).all().item()
    assert all_exactly_k, f"FAIL: some rows ≠ {K} edges"
    print(f"  Fix 2 ✅  every row has exactly {K} active edges\n")

    # Fix 3: movement_only produces sparse mask
    soma_mo = SOMAMask(num_heads=HEADS, seq_len=T, k=K,
                       anticipation_steps=ANTICI, rna_update_interval=5,
                       use_rna=False, use_starling=False)
    for s in range(ANTICI + 5):
        out = soma_mo(torch.rand(HEADS, T, T), step=s)
    sparsity_mv = 1.0 - out.float().mean().item()
    assert sparsity_mv > 0.5, f"FAIL: movement_only mask is dense (sparsity={sparsity_mv:.1%})"
    print(f"  Fix 3 ✅  movement_only sparsity={sparsity_mv:.1%}\n")

    # Fix 4: ema_alpha correctly passed through to MovementPruningMask
    soma_ema = SOMAMask(num_heads=HEADS, seq_len=T, k=K,
                        anticipation_steps=ANTICI, ema_alpha=0.10)
    assert soma_ema.movement.ema_alpha == 0.10, "FAIL: ema_alpha not forwarded"
    print("  Fix 4 ✅  ema_alpha correctly surfaced and forwarded\n")

    # Full SOMA end-to-end
    soma = SOMAMask(num_heads=HEADS, seq_len=T, k=K,
                    anticipation_steps=ANTICI, rna_update_interval=5)
    mask_snapshots = []
    for s in range(40):
        out = soma(torch.rand(HEADS, T, T), step=s)
        if s >= ANTICI:
            mask_snapshots.append(out.clone())

    assert not all(torch.equal(mask_snapshots[0], m) for m in mask_snapshots[1:]), \
        "FAIL: SOMA mask never changed — RNA re-editing is still a no-op."
    sparsity = 1.0 - mask_snapshots[-1].float().mean().item()
    print(f"  SOMA ✅  mask evolves | sparsity={sparsity:.1%}\n")
    print("All architecture smoke tests passed ✅")


_smoke_test()


# ── Commit corrected architecture to GitHub ───────────────────────────────
import os
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"fix: MovementPruningMask EMA + step-wise delta (Eq.1); '
          'StarlingTopologyConstraint exact-k via scatter_; '
          'SOMAMask movement_only branch (was silently dense)"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")

Running architecture smoke tests ...

  Fix 1 ✅  movement_accum evolves after anticipation window (EMA active)

  Fix 2 ✅  every row has exactly 7 active edges

  Fix 3 ✅  movement_only sparsity=78.1%

  Fix 4 ✅  ema_alpha correctly surfaced and forwarded

  SOMA ✅  mask evolves | sparsity=78.1%

All architecture smoke tests passed ✅
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
✅ Pushed to GitHub


Everything up-to-date


## 3. ADAPT-BIO Transformer Architecture

Self-contained architecture — no `src/` dependency needed.  
`SOMAAttention` wraps scaled dot-product attention and gates the weight matrix
with the SOMA boolean mask before softmax.  
The `use_rna` / `use_starling` flags are threaded all the way down to `SOMAMask`
so Section 9 ablations can swap configs without rewriting the model.

In [4]:
class SOMAAttention(nn.Module):
    """
    Multi-head self-attention gated by the SOMA sparsity mask.

    Architecture
    ────────────
    x (B, T, d_model)
      → dropout → QKV linear → (B, H, T, head_dim) × 3
      → scaled dot-product scores  attn  (B, H, T, T)
      → SOMA mask (H, T, T) derived from movement-accumulator signal
      → causal mask intersected with SOMA mask
      → diagonal fallback (self-attention always allowed to prevent NaN)
      → softmax → attention dropout → weighted sum over V
      → out_proj → dropout → (B, T, d_model)

    Causal masking
    ──────────────
    Without a causal mask, position i attends to i+1 and leaks the target token
    into the input. We intersect SOMA mask with a lower-triangular causal mask
    at every forward pass (training and evaluation).

    Diagonal fallback
    ─────────────────
    For early positions (e.g. i=0, only j=0 is causal), SOMA's k=7 columns may
    all be j>i, producing all-inf softmax → NaN. The diagonal fallback ensures
    at least one valid position per row. Self-attention is almost always among
    SOMA's top-k after the first few hundred steps, so this rarely fires.

    Dropout (CRITICAL FIX vs conference paper)
    ─────────────────────────────────────────
    The conference paper's ADAPTBIOTransformer had no dropout, making the
    comparison with FairDenseTransformer (dropout=0.3) unfair. This version
    adds embedding dropout and FF dropout matching the dense baseline exactly.
    The SOMA sparsity itself acts as structural regularisation, but explicit
    dropout is still added so the comparison is controlled.
    """
    def __init__(self, d_model, num_heads, seq_len, k=7,
                 anticipation_steps=100, rna_update_interval=500,
                 ema_alpha=0.05, dropout=0.0,
                 use_rna=True, use_starling=True):
        super().__init__()
        assert d_model % num_heads == 0, \
            f"d_model={d_model} must be divisible by num_heads={num_heads}"
        self.num_heads   = num_heads
        self.head_dim    = d_model // num_heads
        self.scale       = self.head_dim ** -0.5
        self.qkv         = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj    = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop   = nn.Dropout(dropout)   # attention weight dropout
        self.resid_drop  = nn.Dropout(dropout)   # residual dropout after projection
        self.soma        = SOMAMask(
            num_heads=num_heads, seq_len=seq_len, k=k,
            anticipation_steps=anticipation_steps,
            rna_update_interval=rna_update_interval,
            ema_alpha=ema_alpha,
            use_rna=use_rna, use_starling=use_starling,
        )

    def forward(self, x: torch.Tensor, step: int = 0) -> torch.Tensor:
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) * self.scale   # (B, H, T, T)

        # ── Causal mask ───────────────────────────────────────────────────
        # float mask: 0 = keep, -inf = block (avoids bool→float conversion
        # ambiguity in future PyTorch versions)
        causal = torch.zeros(T, T, device=x.device)
        causal.masked_fill_(
            torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool),
                       diagonal=1),
            float('-inf')
        )                                                # (T, T)

        # ── SOMA sparsity mask ────────────────────────────────────────────
        soma_signal = attn.detach().mean(0)              # (H, T, T)
        soma_mask   = self.soma(soma_signal, step=step)  # (H, T, T)  True=keep

        # ── Diagonal fallback ─────────────────────────────────────────────
        diag    = torch.eye(T, dtype=torch.bool, device=x.device)
        allowed = (soma_mask | diag.unsqueeze(0))        # (H, T, T)

        # ── Apply combined mask ───────────────────────────────────────────
        # First apply SOMA+diagonal (bool → additive -inf for blocked positions)
        attn = attn.masked_fill(~allowed.unsqueeze(0), float('-inf'))
        # Then apply causal mask additively
        attn = attn + causal.unsqueeze(0).unsqueeze(0)  # broadcast (1,1,T,T)
        attn = F.softmax(attn, dim=-1)
        attn = torch.nan_to_num(attn, nan=0.0)
        attn = self.attn_drop(attn)                      # attention dropout

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.out_proj(out))       # residual dropout


class ADAPTBIOBlock(nn.Module):
    def __init__(self, d_model, num_heads, seq_len, k=7,
                 anticipation_steps=100, rna_update_interval=500,
                 ema_alpha=0.05, dropout=0.0,
                 use_rna=True, use_starling=True):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = SOMAAttention(d_model, num_heads, seq_len, k=k,
                                   anticipation_steps=anticipation_steps,
                                   rna_update_interval=rna_update_interval,
                                   ema_alpha=ema_alpha, dropout=dropout,
                                   use_rna=use_rna, use_starling=use_starling)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Dropout(dropout),                         # FF dropout
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, step: int) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), step=step)
        x = x + self.ff(self.norm2(x))
        return x


class ADAPTBIOTransformer(nn.Module):
    """
    ADAPT-BIO autoregressive transformer.

    dropout=0.3 matches FairDenseTransformer's regularisation exactly,
    ensuring all PPL comparisons are controlled (CRITICAL FIX: conference
    paper's ADAPTBIOTransformer had no dropout anywhere).
    """
    def __init__(self, vocab_size, d_model, num_heads, num_layers, seq_len,
                 k=7, anticipation_steps=100, rna_update_interval=500,
                 ema_alpha=0.05, dropout=0.3,
                 use_rna=True, use_starling=True):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model)
        self.pos_enc = nn.Embedding(seq_len, d_model)
        self.drop    = nn.Dropout(dropout)               # embedding dropout
        self.blocks  = nn.ModuleList([
            ADAPTBIOBlock(d_model, num_heads, seq_len, k=k,
                          anticipation_steps=anticipation_steps,
                          rna_update_interval=rna_update_interval,
                          ema_alpha=ema_alpha, dropout=dropout,
                          use_rna=use_rna, use_starling=use_starling)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x: torch.Tensor, step: int = 0) -> torch.Tensor:
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        h    = self.drop(self.embed(x) + self.pos_enc(pos))   # embedding dropout
        for block in self.blocks:
            h = block(h, step=step)
        return self.head(self.norm(h))


class FairDenseTransformer(nn.Module):
    """
    Dense baseline. Causal mask passed as additive float mask (not bool)
    to avoid dtype-dependent behaviour in future PyTorch versions.
    """
    def __init__(self, vocab_size, d_model, num_heads, num_layers, seq_len, dropout=0.3):
        super().__init__()
        self.embed    = nn.Embedding(vocab_size, d_model)
        self.pos_enc  = nn.Embedding(seq_len, d_model)
        self.drop     = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_model * 4, dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x: torch.Tensor, step=None) -> torch.Tensor:
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        h    = self.drop(self.embed(x) + self.pos_enc(pos))

        # ── Causal mask as additive float (not bool) ──────────────────────
        # nn.TransformerEncoderLayer adds this to attention logits before
        # softmax; -inf blocks future positions.
        causal_mask = torch.zeros(T, T, device=x.device)
        causal_mask.masked_fill_(
            torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool),
                       diagonal=1),
            float('-inf')
        )
        h = self.transformer(h, mask=causal_mask)
        return self.head(self.drop(h))


# ── Smoke tests ───────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Test 1: ADAPTBIOTransformer forward pass shape
_m = ADAPTBIOTransformer(vocab_size=1000, d_model=64, num_heads=4,
                          num_layers=2, seq_len=32, k=7, dropout=0.3).to(device)
_x = torch.randint(0, 1000, (2, 32), device=device)
_out = _m(_x, step=150)
assert _out.shape == (2, 32, 1000)
print(f"ADAPTBIOTransformer ✅  shape={_out.shape}  params={sum(p.numel() for p in _m.parameters()):,}")
del _m, _x, _out

# Test 2: FairDenseTransformer forward pass + causal mask dtype is float
_d = FairDenseTransformer(vocab_size=1000, d_model=64, num_heads=4,
                           num_layers=2, seq_len=32).to(device)
_x = torch.randint(0, 1000, (2, 32), device=device)
_out = _d(_x)
assert _out.shape == (2, 32, 1000)
print(f"FairDenseTransformer  ✅  shape={_out.shape}")
del _d, _x, _out

# Test 3: Dropout is actually present in ADAPTBIOTransformer
_m_drop = ADAPTBIOTransformer(vocab_size=100, d_model=32, num_heads=2,
                               num_layers=1, seq_len=16, k=3, dropout=0.5)
has_drop = any(isinstance(m, nn.Dropout) for m in _m_drop.modules())
assert has_drop, "FAIL: no Dropout found in ADAPTBIOTransformer — dropout fix not applied"
print("Dropout check         ✅  ADAPTBIOTransformer contains Dropout layers")
del _m_drop

# Test 4: Causal leak check — output at pos=0 must not change when pos=1 is perturbed
_attn = SOMAAttention(d_model=32, num_heads=2, seq_len=8, k=3,
                       anticipation_steps=2, rna_update_interval=100,
                       dropout=0.0).to(device)
_attn.eval()
_x_in = torch.randn(1, 8, 32, device=device)
_x_a  = _x_in.clone(); _x_a[0, 1, :] += 1.0
with torch.no_grad():
    _out_orig = _attn(_x_in, step=5)
    _out_perturbed = _attn(_x_a, step=5)
max_leak = (_out_perturbed[0, 0] - _out_orig[0, 0]).abs().max().item()
assert max_leak < 1e-5, f"FAIL: causal leakage detected Δ={max_leak:.4f}"
print(f"Causal leak check     ✅  max Δ at pos=0 from perturbing pos=1: {max_leak:.2e}\n")
del _attn, _x_in, _x_a, _out_orig, _out_perturbed

print("All architecture smoke tests passed ✅")


ADAPTBIOTransformer ✅  shape=torch.Size([2, 32, 1000])  params=229,632
FairDenseTransformer  ✅  shape=torch.Size([2, 32, 1000])
Dropout check         ✅  ADAPTBIOTransformer contains Dropout layers
Causal leak check     ✅  max Δ at pos=0 from perturbing pos=1: 0.00e+00

All architecture smoke tests passed ✅


## 4. Data Loading

WikiText-2 is loaded fully into memory (small — ~2M tokens).  
WikiText-103 uses a chunked `Dataset` with a token cap to avoid OOM on Kaggle.

In [5]:
import math, gc, time
from transformers import AutoTokenizer
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer.pad_token = tokenizer.eos_token if tokenizer.pad_token is None else tokenizer.pad_token
VOCAB_SIZE = tokenizer.vocab_size   # 30522
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Vocab : {VOCAB_SIZE}")
print(f"Device: {DEVICE}")


class WikiText2Dataset(Dataset):
    def __init__(self, split, seq_len=128):
        raw     = load_dataset("wikitext", "wikitext-2-raw-v1", split=split)
        all_ids = []
        for item in raw:
            t = item["text"].strip()
            if t:
                all_ids.extend(tokenizer.encode(t, add_special_tokens=False))
        n = (len(all_ids) // (seq_len + 1)) * (seq_len + 1)
        self.chunks = torch.tensor(all_ids[:n], dtype=torch.long).view(-1, seq_len + 1)
        print(f"  WikiText-2 [{split}]: {len(self.chunks)} chunks (seq_len={seq_len})")
    def __len__(self):        return len(self.chunks)
    def __getitem__(self, i): return self.chunks[i]


class WikiText103Dataset(Dataset):
    def __init__(self, split, n_chunks=20000, seq_len=128):
        raw     = load_dataset("wikitext", "wikitext-103-raw-v1",
                               split=split, trust_remote_code=True)
        all_ids = []
        for item in raw:
            t = item["text"].strip()
            if not t:
                continue
            all_ids.extend(tokenizer.encode(t, add_special_tokens=False))
            if len(all_ids) >= n_chunks * (seq_len + 1):
                break
        n = (len(all_ids) // (seq_len + 1)) * (seq_len + 1)
        self.chunks = torch.tensor(all_ids[:n], dtype=torch.long).view(-1, seq_len + 1)
        print(f"  WikiText-103 [{split}]: {len(self.chunks)} chunks (seq_len={seq_len})")
    def __len__(self):        return len(self.chunks)
    def __getitem__(self, i): return self.chunks[i]


def make_loaders(dataset_cls, seq_len=128, batch_size=32, **dataset_kwargs):
   
    tr = dataset_cls("train",      seq_len=seq_len, **dataset_kwargs)
    va = dataset_cls("validation", seq_len=seq_len, **dataset_kwargs)
    return (DataLoader(tr, batch_size=batch_size, shuffle=True,  drop_last=True),
            DataLoader(va, batch_size=batch_size, shuffle=False, drop_last=True))


SEQ_LEN    = 128
BATCH_SIZE = 32

print("\nLoading WikiText-2...")
wt2_train, wt2_val = make_loaders(WikiText2Dataset, seq_len=SEQ_LEN, batch_size=BATCH_SIZE)
print(f"  Train batches: {len(wt2_train)} | Val batches: {len(wt2_val)}")
print("Data loaders ready ✅")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocab : 30522
Device: cuda

Loading WikiText-2...


...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (645 > 512). Running this sequence through the model will result in indexing errors


  WikiText-2 [train]: 17858 chunks (seq_len=128)
  WikiText-2 [validation]: 1850 chunks (seq_len=128)
  Train batches: 558 | Val batches: 57
Data loaders ready ✅


## 5. Training & Benchmarking Utilities

- `evaluate(model, loader)` — returns perplexity  
- `train_run(...)` — single training run, returns (final val PPL, wall time)  
- `theoretical_flops(k, seq_len)` — dense vs sparse attention FLOPs  
- `benchmark_timing(...)` — wall-clock train/inference time + peak GPU memory

In [6]:
import numpy as np
criterion = nn.CrossEntropyLoss()

# ── Sentinel step value for evaluation ───────────────────────────────────────
# Using float('inf') is cleaner than a magic integer: it will always be
# >= anticipation_steps regardless of how that hyperparameter is set.
# SOMAMask.forward checks `step < anticipation_steps`, so inf → sparse mode.
EVAL_STEP = 10**9   # int sentinel; torch doesn't accept float for step


def evaluate(model, loader, max_batches=200):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= max_batches: break
            batch    = batch.to(DEVICE)
            inp, tgt = batch[:, :-1], batch[:, 1:]
            logits   = model(inp, step=EVAL_STEP)   # always sparse during eval
            loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
            total_loss   += loss.item() * tgt.numel()
            total_tokens += tgt.numel()
    return math.exp(total_loss / total_tokens)


def train_run(model, train_loader, val_loader, steps=10000,
              lr=3e-4, log_every=1000, label=""):
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
    model.train()
    it = iter(train_loader)
    t0 = time.time()
    for step in range(steps):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(train_loader)
            batch = next(it)
        batch    = batch.to(DEVICE)
        inp, tgt = batch[:, :-1], batch[:, 1:]
        logits   = model(inp, step=step)
        loss     = criterion(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()
        if step % log_every == 0:
            elapsed = time.time() - t0
            print(f"  [{label}] step {step:5d}/{steps} | "
                  f"loss={loss.item():.4f} | elapsed={elapsed:.0f}s")
    val_ppl = evaluate(model, val_loader)
    print(f"  [{label}] ✅ Final Val PPL = {val_ppl:.4f}  "
          f"(total {time.time()-t0:.0f}s)")
    return val_ppl, time.time() - t0


def theoretical_flops(k, seq_len):
    dense  = seq_len * seq_len
    sparse = seq_len * k
    return {
        "dense_attn_flops":  dense,
        "sparse_attn_flops": sparse,
        "flop_reduction":    1.0 - sparse / dense,
        "flops_ratio":       dense / sparse,
    }


def benchmark_timing(model, seq_len=128, batch_size=8,
                     n_warmup=20, n_train_steps=100, n_infer_steps=100, label=""):
   
    vocab = model.head.out_features
    x = torch.randint(0, vocab, (batch_size, seq_len), device=DEVICE)
    y = torch.randint(0, vocab, (batch_size, seq_len), device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

    # ── Warmup ───────────────────────────────────────────────────────────────
    model.train()
    for step in range(n_warmup):
        logits = model(x, step=step)
        loss   = criterion(logits.reshape(-1, vocab), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(DEVICE)

    # ── Training timing ───────────────────────────────────────────────────────
    model.train()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for step in range(n_train_steps):
        logits = model(x, step=n_warmup + step)
        loss   = criterion(logits.reshape(-1, vocab), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    train_ms = (time.time() - t0) / n_train_steps * 1000

    # ── Inference timing ──────────────────────────────────────────────────────
    model.eval()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(n_infer_steps):
            _ = model(x, step=EVAL_STEP)
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    infer_ms = (time.time() - t0) / n_infer_steps * 1000

    peak_mb = (torch.cuda.max_memory_allocated(DEVICE) / 1024**2
               if DEVICE.type == "cuda" else float("nan"))

    print(f"  [{label}] train {train_ms:.1f} ms/step | "
          f"infer {infer_ms:.1f} ms/step | "
          f"peak GPU {peak_mb:.1f} MB")
    return {"train_ms_per_step": round(train_ms, 2),
            "infer_ms_per_step": round(infer_ms, 2),
            "peak_gpu_mb":       round(peak_mb, 1)}


print("Utilities defined ✅")

Utilities defined ✅


## 6. Main Results — Dense vs ADAPT-BIO

3 seeds × 2 model sizes (d_model=128, d_model=256) on WikiText-2.  
Each run is 10000 steps. Results saved to `results.json` after each size — 
if the kernel restarts, the completed size won't re-run.

In [7]:
 
import numpy as np
import gc
 
STEPS = 10000
SEEDS = [42, 123, 7]
 
 
def run_seeds(ModelClass, label, seeds,
              loader_train, loader_val,        # ← now explicit, not closed-over
              steps=STEPS, **model_kwargs):
    """
    Train `ModelClass(**model_kwargs)` for each seed and return all PPLs.
 
    Parameters
    ----------
    ModelClass   : class to instantiate (FairDenseTransformer or ADAPTBIOTransformer)
    label        : string prefix for progress logs
    seeds        : list of int random seeds
    loader_train : DataLoader — the training split for this experiment
    loader_val   : DataLoader — the validation split for this experiment
    steps        : number of training steps per seed
    **model_kwargs : forwarded to ModelClass.__init__
    """
    ppls = []
    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)
        model = ModelClass(**model_kwargs).to(DEVICE)
        ppl, _ = train_run(
            model, loader_train, loader_val,   # ← explicit loaders
            steps=steps, label=f"{label}-s{seed}"
        )
        ppls.append(round(ppl, 4))
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return ppls
 
 
# ── d_model = 128  (WikiText-2) ─────────────────────────────────────────────
if 'main_wt2_128' not in results:
    print("=" * 55)
    print("  WikiText-2 | d_model = 128")
    print("=" * 55)
 
    dense_ppls = run_seeds(
        FairDenseTransformer, "Dense-128", SEEDS,
        loader_train=wt2_train, loader_val=wt2_val,   # ← explicit
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN)
 
    adapt_ppls = run_seeds(
        ADAPTBIOTransformer, "ADAPT-128", SEEDS,
        loader_train=wt2_train, loader_val=wt2_val,   # ← explicit
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=500)
 
    results['main_wt2_128'] = {
        'dense': {'ppls': dense_ppls,
                  'mean': round(float(np.mean(dense_ppls)), 4),
                  'std':  round(float(np.std(dense_ppls)),  4)},
        'adapt': {'ppls': adapt_ppls,
                  'mean': round(float(np.mean(adapt_ppls)), 4),
                  'std':  round(float(np.std(adapt_ppls)),  4)},
        'flops': theoretical_flops(7, SEQ_LEN),
    }
    save_results()
else:
    print("main_wt2_128 already done — skipping.")
 
r = results['main_wt2_128']
print(f"\n  Dense-128 : PPL = {r['dense']['mean']:.2f} ± {r['dense']['std']:.2f}")
print(f"  ADAPT-128 : PPL = {r['adapt']['mean']:.2f} ± {r['adapt']['std']:.2f}")
print(f"  Sparsity  : 94.5% | FLOPs reduction:"
      f" {r['flops']['flop_reduction']*100:.1f}% ({r['flops']['flops_ratio']:.1f}×)")
 
 
# ── d_model = 256  (WikiText-2) ─────────────────────────────────────────────
if 'main_wt2_256' not in results:
    print("\n" + "=" * 55)
    print("  WikiText-2 | d_model = 256")
    print("=" * 55)
 
    dense_ppls = run_seeds(
        FairDenseTransformer, "Dense-256", SEEDS,
        loader_train=wt2_train, loader_val=wt2_val,
        vocab_size=VOCAB_SIZE, d_model=256, num_heads=8,
        num_layers=2, seq_len=SEQ_LEN)
 
    adapt_ppls = run_seeds(
        ADAPTBIOTransformer, "ADAPT-256", SEEDS,
        loader_train=wt2_train, loader_val=wt2_val,
        vocab_size=VOCAB_SIZE, d_model=256, num_heads=8,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=500)
 
    results['main_wt2_256'] = {
        'dense': {'ppls': dense_ppls,
                  'mean': round(float(np.mean(dense_ppls)), 4),
                  'std':  round(float(np.std(dense_ppls)),  4)},
        'adapt': {'ppls': adapt_ppls,
                  'mean': round(float(np.mean(adapt_ppls)), 4),
                  'std':  round(float(np.std(adapt_ppls)),  4)},
        'flops': theoretical_flops(7, SEQ_LEN),
    }
    save_results()
else:
    print("main_wt2_256 already done — skipping.")
 
r2 = results['main_wt2_256']
print(f"\n  Dense-256 : PPL = {r2['dense']['mean']:.2f} ± {r2['dense']['std']:.2f}")
print(f"  ADAPT-256 : PPL = {r2['adapt']['mean']:.2f} ± {r2['adapt']['std']:.2f}")
print(f"  Sparsity  : 94.5% | FLOPs reduction:"
      f" {r2['flops']['flop_reduction']*100:.1f}% ({r2['flops']['flops_ratio']:.1f}×)")
 
# ── Commit WikiText-2 main results ────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: WikiText-2 main results d=128 + d=256 (3 seeds, EMA-fixed)"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")
 

  WikiText-2 | d_model = 128
  [Dense-128-s42] step     0/10000 | loss=10.5460 | elapsed=0s
  [Dense-128-s42] step  1000/10000 | loss=6.8612 | elapsed=64s
  [Dense-128-s42] step  2000/10000 | loss=6.6336 | elapsed=134s
  [Dense-128-s42] step  3000/10000 | loss=6.5626 | elapsed=203s
  [Dense-128-s42] step  4000/10000 | loss=6.3500 | elapsed=273s
  [Dense-128-s42] step  5000/10000 | loss=6.3047 | elapsed=342s
  [Dense-128-s42] step  6000/10000 | loss=6.1026 | elapsed=412s
  [Dense-128-s42] step  7000/10000 | loss=6.3004 | elapsed=481s
  [Dense-128-s42] step  8000/10000 | loss=6.0957 | elapsed=551s
  [Dense-128-s42] step  9000/10000 | loss=6.2276 | elapsed=620s
  [Dense-128-s42] ✅ Final Val PPL = 491.0867  (total 690s)
  [Dense-128-s123] step     0/10000 | loss=10.5570 | elapsed=0s
  [Dense-128-s123] step  1000/10000 | loss=6.9224 | elapsed=69s
  [Dense-128-s123] step  2000/10000 | loss=6.5562 | elapsed=138s
  [Dense-128-s123] step  3000/10000 | loss=6.5082 | elapsed=206s
  [Dense-128-s12

Everything up-to-date


## 7. k-Sweep Ablation

Validates the starling k=7 choice empirically.  
5 values of k, 1 seed each, 10000 steps on WikiText-2 (d_model=128).

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

K_VALS = [3, 5, 7, 9, 12]

if 'k_sweep' not in results:
    print("Running k-sweep (5 runs × 10000 steps)...")
    k_results = {}
    for kv in K_VALS:
        torch.manual_seed(42); np.random.seed(42)
        model = ADAPTBIOTransformer(
            vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
            num_layers=2, seq_len=SEQ_LEN, k=kv,
            anticipation_steps=100, rna_update_interval=500
        ).to(DEVICE)
        ppl, _ = train_run(model, wt2_train, wt2_val,
                           steps=STEPS, label=f"k={kv}")
        flops = theoretical_flops(kv, SEQ_LEN)
        k_results[str(kv)] = {
            'ppl':            round(ppl, 4),
            'sparsity_pct':   round(flops['flop_reduction'] * 100, 2),
            'flops_ratio':    round(flops['flops_ratio'], 2),
        }
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    results['k_sweep'] = k_results
    save_results()
else:
    print("k_sweep already done — skipping.")

# ── Print table ───────────────────────────────────────────────
print(f"\n  {'k':>4} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11}")
print("  " + "-"*40)
for kv in K_VALS:
    r = results['k_sweep'][str(kv)]
    print(f"  {kv:>4} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | {r['flops_ratio']:>9.1f}×")

# ── Figure ────────────────────────────────────────────────────
ppls = [results['k_sweep'][str(k)]['ppl']         for k in K_VALS]
spar = [results['k_sweep'][str(k)]['sparsity_pct'] for k in K_VALS]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.plot(K_VALS, ppls, 'o-', color='#2196F3', lw=2.5, ms=8, label='Val PPL ↓')
ax1.axvline(x=7, color='red', ls='--', lw=1.5, label='k=7 (chosen operating point)')
ax2.plot(K_VALS, spar, 's--', color='#4CAF50', lw=2, ms=8, label='Sparsity %')
ax1.set_xlabel('k (neighbours per token)', fontsize=12)
ax1.set_ylabel('Validation Perplexity ↓', color='#2196F3', fontsize=12)
ax2.set_ylabel('Attention Sparsity % ↑', color='#4CAF50', fontsize=12)
ax2.set_ylim(85, 100)
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc='upper right', fontsize=10)
plt.title('Figure 3: k-Sweep — Starling Neighbour Constraint Validation\n'
          'ADAPT-BIO on WikiText-2 (10000 steps, d_model=128)',
          fontsize=12, fontweight='bold')
plt.xticks(K_VALS)
plt.tight_layout()
os.makedirs('/kaggle/working/adapt-bio/paper/figures', exist_ok=True)
plt.savefig('/kaggle/working/adapt-bio/paper/figures/k_sweep_final.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 3 saved → paper/figures/k_sweep_final.png")


# ── Pareto frontier analysis ──────────────────────────────────
sweep = results['k_sweep']
K_VALS_P = sorted(int(k) for k in sweep.keys())

def is_pareto_optimal(k_candidate, sw, k_vals):
    ppl_c  = sw[str(k_candidate)]['ppl']
    spar_c = sw[str(k_candidate)]['sparsity_pct']
    for k_other in k_vals:
        if k_other == k_candidate:
            continue
        ppl_o  = sw[str(k_other)]['ppl']
        spar_o = sw[str(k_other)]['sparsity_pct']
        if ppl_o < ppl_c and spar_o >= spar_c:
            return False
    return True

pareto_ks  = [k for k in K_VALS_P if is_pareto_optimal(k, sweep, K_VALS_P)]
chosen_k   = max(pareto_ks, key=lambda k: sweep[str(k)]['sparsity_pct'])
chosen_ppl  = sweep[str(chosen_k)]['ppl']
chosen_spar = sweep[str(chosen_k)]['sparsity_pct']

print("=" * 60)
print("  PARETO FRONTIER ANALYSIS — k-sweep")
print("=" * 60)
print(f"\n  Pareto-optimal k values : {pareto_ks}")
print(f"  Chosen operating point  : k={chosen_k} "
      f"(PPL={chosen_ppl:.2f}, Sparsity={chosen_spar:.1f}%)\n")

print(f"  {'k':>4} | {'PPL':>8} | {'Sparsity':>9} | {'Status':>25}")
print("  " + "-" * 52)
for kv in K_VALS_P:
    r = sweep[str(kv)]
    if kv == chosen_k:
        status = "← chosen operating point"
    elif kv in pareto_ks:
        status = "Pareto-optimal"
    else:
        status = "dominated"
    print(f"  {kv:>4} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | {status:>25}")

# Find what dominates chosen_k, if anything
dominated_by = [
    kv for kv in K_VALS_P if kv != chosen_k
    and sweep[str(kv)]['ppl'] < chosen_ppl
    and sweep[str(kv)]['sparsity_pct'] >= chosen_spar
]

if dominated_by:
    print(f"\n  ⚠️  k={dominated_by} dominates chosen k={chosen_k} "
          f"— operating point needs revision.")
else:
    print(f"\n  ✅ k={chosen_k} is Pareto-optimal — no k achieves both "
          f"lower PPL and equal or higher sparsity.")

results['pareto_analysis'] = {
    'pareto_ks':     pareto_ks,
    'chosen_k':      chosen_k,
    'chosen_ppl':    chosen_ppl,
    'chosen_spar':   chosen_spar,
    'dominated_by':  dominated_by,
    'pareto_optimal': len(dominated_by) == 0,
}
save_results()
print("\n✅ Pareto analysis saved to results.json")
# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: k-sweep ablation + Figure 3"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")

Running k-sweep (5 runs × 10000 steps)...
  [k=3] step     0/10000 | loss=10.4922 | elapsed=0s
  [k=3] step  1000/10000 | loss=6.7242 | elapsed=66s
  [k=3] step  2000/10000 | loss=6.5091 | elapsed=132s
  [k=3] step  3000/10000 | loss=6.2375 | elapsed=199s
  [k=3] step  4000/10000 | loss=6.0878 | elapsed=266s
  [k=3] step  5000/10000 | loss=6.1317 | elapsed=332s
  [k=3] step  6000/10000 | loss=6.0004 | elapsed=399s
  [k=3] step  7000/10000 | loss=5.9570 | elapsed=465s
  [k=3] step  8000/10000 | loss=5.9619 | elapsed=532s
  [k=3] step  9000/10000 | loss=5.9674 | elapsed=599s
  [k=3] ✅ Final Val PPL = 538.6089  (total 667s)
  [k=5] step     0/10000 | loss=10.4922 | elapsed=0s
  [k=5] step  1000/10000 | loss=6.7236 | elapsed=66s
  [k=5] step  2000/10000 | loss=6.5077 | elapsed=132s
  [k=5] step  3000/10000 | loss=6.2346 | elapsed=199s
  [k=5] step  4000/10000 | loss=6.0859 | elapsed=265s
  [k=5] step  5000/10000 | loss=6.1239 | elapsed=332s
  [k=5] step  6000/10000 | loss=5.9959 | elapsed=

Everything up-to-date


## 8. RNA Interval Sweep — Validating Dynamic Mask Revision

Tests 5 values of Δ (update interval) + a Frozen baseline (no RNA editing).  
The critical result: Δ=500 should beat Frozen — this is the direct empirical proof  
that dynamic mid-training mask revision (ADAPT-BIO's core contribution) outperforms  
the static-mask regime of all prior work.

In [9]:
DELTA_VALS  = [50, 200, 500, 2000]
FROZEN_KEY  = "frozen"

if 'rna_sweep' not in results:
    print("Running RNA interval sweep (5 runs × 10000 steps)...")
    rna_results = {}

    # ── Dynamic Δ runs ────────────────────────────────────────
    for delta in DELTA_VALS:
        torch.manual_seed(42); np.random.seed(42)
        model = ADAPTBIOTransformer(
            vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
            num_layers=2, seq_len=SEQ_LEN, k=7,
            anticipation_steps=100, rna_update_interval=delta
        ).to(DEVICE)
        ppl, _ = train_run(model, wt2_train, wt2_val,
                           steps=STEPS, label=f"Δ={delta}")
        rna_results[str(delta)] = round(ppl, 4)
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # ── Frozen baseline (RNA disabled — use_rna=False) ────────
    torch.manual_seed(42); np.random.seed(42)
    model = ADAPTBIOTransformer(
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=999999,
        use_rna=False
    ).to(DEVICE)
    ppl, _ = train_run(model, wt2_train, wt2_val,
                       steps=STEPS, label="Frozen")
    rna_results[FROZEN_KEY] = round(ppl, 4)
    del model; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['rna_sweep'] = rna_results
    save_results()
else:
    print("rna_sweep already done — skipping.")

# ── Print table ───────────────────────────────────────────────
frozen_ppl = results['rna_sweep'][FROZEN_KEY]
print(f"\n  {'Δ (steps)':>12} | {'Val PPL':>8} | Note")
print("  " + "-" * 45)
for delta in DELTA_VALS:
    ppl = results['rna_sweep'][str(delta)]
    beat = "✅ beats frozen" if ppl < frozen_ppl else "❌ worse than frozen"
    print(f"  {delta:>12} | {ppl:>8.2f} | {beat}")
print(f"  {'Frozen':>12} | {frozen_ppl:>8.2f} | static mask baseline")

# ── Figure ────────────────────────────────────────────────────
x_labels = [str(d) for d in DELTA_VALS] + ["Frozen"]
y_vals    = [results['rna_sweep'][str(d)] for d in DELTA_VALS] + [frozen_ppl]
colors    = ['#FF7043' if v >= frozen_ppl else '#4CAF50' for v in y_vals[:-1]] + ['#9E9E9E']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(x_labels, y_vals, color=colors, width=0.6, edgecolor='black', linewidth=0.8)
ax.axhline(y=frozen_ppl, color='black', ls='--', lw=1.5,
           label=f'Frozen baseline (PPL={frozen_ppl:.2f})')

# Annotate bars
for bar, val in zip(bars, y_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('RNA Update Interval Δ (steps)', fontsize=12)
ax.set_ylabel('Validation Perplexity ↓', fontsize=12)
ax.legend(fontsize=10)
plt.title('Figure 4: RNA Interval Sweep — Octopus Self-Editing Validation\n'
          'ADAPT-BIO on WikiText-2 (k=7, 10000 steps, d_model=128)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/rna_sweep_final.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 4 saved → paper/figures/rna_sweep_final.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: RNA interval sweep + Figure 4"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")

Running RNA interval sweep (5 runs × 10000 steps)...
  [Δ=50] step     0/10000 | loss=10.4922 | elapsed=0s
  [Δ=50] step  1000/10000 | loss=6.7193 | elapsed=66s
  [Δ=50] step  2000/10000 | loss=6.5080 | elapsed=133s
  [Δ=50] step  3000/10000 | loss=6.2367 | elapsed=199s
  [Δ=50] step  4000/10000 | loss=6.0830 | elapsed=265s
  [Δ=50] step  5000/10000 | loss=6.1235 | elapsed=332s
  [Δ=50] step  6000/10000 | loss=6.0011 | elapsed=399s
  [Δ=50] step  7000/10000 | loss=5.9552 | elapsed=465s
  [Δ=50] step  8000/10000 | loss=5.9613 | elapsed=531s
  [Δ=50] step  9000/10000 | loss=5.9570 | elapsed=598s
  [Δ=50] ✅ Final Val PPL = 535.6475  (total 666s)
  [Δ=200] step     0/10000 | loss=10.4922 | elapsed=0s
  [Δ=200] step  1000/10000 | loss=6.7222 | elapsed=66s
  [Δ=200] step  2000/10000 | loss=6.5123 | elapsed=132s
  [Δ=200] step  3000/10000 | loss=6.2370 | elapsed=198s
  [Δ=200] step  4000/10000 | loss=6.0820 | elapsed=264s
  [Δ=200] step  5000/10000 | loss=6.1201 | elapsed=330s
  [Δ=200] step 

Everything up-to-date


## 9. Component Ablation — SOMA Decomposition

Isolates the contribution of each biological signal by disabling one at a time:

| Config | Movement Pruning | RNA Update | k=7 Starling |
|---|---|---|---|
| Full ADAPT-BIO | ✓ | ✓ | ✓ |
| No-RNA | ✓ | ✗ (frozen after anticipation) | ✓ |
| No-Starling | ✓ | ✓ | ✗ (dense k=T) |

Reports: Val PPL, Attention Sparsity, Theoretical FLOPs reduction.  
> **Scale note:** At d_model=128 / WikiText-2, RNA's benefit is masked by signal noise  
> (consistent with the scale sensitivity observed in Section 8). The ablation here  
> quantifies the structural contribution of each component at this scale.

In [10]:
ABLATION_CONFIGS = {
    'full':           dict(use_rna=True,  use_starling=True),
    'no_rna':         dict(use_rna=False, use_starling=True),
    'no_starling':    dict(use_rna=True,  use_starling=False),
    'movement_only':  dict(use_rna=False, use_starling=False),
}

if 'component_ablation' not in results:
    print(f"Running component ablation ({len(ABLATION_CONFIGS)} configs × {STEPS} steps)...")
    ablation_results = {}

    for name, flags in ABLATION_CONFIGS.items():
        torch.manual_seed(42); np.random.seed(42)
        k_val = 7 if flags['use_starling'] else SEQ_LEN

        model = ADAPTBIOTransformer(
            vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
            num_layers=2, seq_len=SEQ_LEN, k=k_val, dropout=0.3,
            anticipation_steps=100, rna_update_interval=500,
            **flags
        ).to(DEVICE)
        ppl, wall_time = train_run(model, wt2_train, wt2_val,
                                   steps=STEPS, label=name)

        flops = theoretical_flops(k_val, SEQ_LEN)
        ablation_results[name] = {
            'ppl':          round(ppl, 4),
            'sparsity_pct': round(flops['flop_reduction'] * 100, 2),
            'flops_ratio':  round(flops['flops_ratio'], 2),
            'wall_time_s':  round(wall_time, 1),
        }
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['component_ablation'] = ablation_results
    save_results()
else:
    print("component_ablation already done — skipping.")

# ── Print table — all 4 configs ──────────────────────────────────────────────
DISPLAY_ORDER  = ['movement_only', 'no_rna', 'no_starling', 'full']
DISPLAY_LABELS = {
    'full':          'Full ADAPT-BIO',
    'no_rna':        'No-RNA',
    'no_starling':   'No-Starling',
    'movement_only': 'Movement-Only',
}
print(f"\n  {'Config':<18} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11} | {'Time (s)':>9}")
print("  " + "-" * 65)
for name in DISPLAY_ORDER:
    r = results['component_ablation'][name]
    print(f"  {DISPLAY_LABELS[name]:<18} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% "
          f"| {r['flops_ratio']:>9.1f}× | {r['wall_time_s']:>9.1f}")

# ── Figure 5 — all 4 configs ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
configs    = DISPLAY_ORDER
labels     = [DISPLAY_LABELS[c] for c in configs]
bar_colors = ['#B0BEC5', '#FF7043', '#FFC107', '#4CAF50']

# Panel A — PPL
ppls = [results['component_ablation'][c]['ppl'] for c in configs]
axes[0].bar(labels, ppls, color=bar_colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(ppls):
    axes[0].text(i, v + 1, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Validation Perplexity ↓', fontsize=11)
axes[0].set_title('(A) PPL by Component', fontweight='bold')
axes[0].tick_params(axis='x', labelsize=8)

# Panel B — Sparsity
spars = [results['component_ablation'][c]['sparsity_pct'] for c in configs]
axes[1].bar(labels, spars, color=bar_colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(spars):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Attention Sparsity % ↑', fontsize=11)
axes[1].set_ylim(0, 105)
axes[1].set_title('(B) Sparsity by Component', fontweight='bold')
axes[1].tick_params(axis='x', labelsize=8)

# Panel C — FLOPs ratio
flops_r = [results['component_ablation'][c]['flops_ratio'] for c in configs]
axes[2].bar(labels, flops_r, color=bar_colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(flops_r):
    axes[2].text(i, v + 0.2, f'{v:.1f}×', ha='center', fontsize=10, fontweight='bold')
axes[2].set_ylabel('FLOPs Reduction (×) ↑', fontsize=11)
axes[2].set_title('(C) FLOPs Ratio by Component', fontweight='bold')
axes[2].tick_params(axis='x', labelsize=8)

fig.suptitle('Figure 5: SOMA Component Ablation (4 configs)\n'
             'ADAPT-BIO on WikiText-2 (d_model=128, 10000 steps)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('/kaggle/working/adapt-bio/paper/figures', exist_ok=True)
plt.savefig('/kaggle/working/adapt-bio/paper/figures/component_ablation.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("✅ Figure 5 saved → paper/figures/component_ablation.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: component ablation (mentor req 1) + Figure 5"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")

Running component ablation (4 configs × 10000 steps)...
  [full] step     0/10000 | loss=10.4922 | elapsed=0s
  [full] step  1000/10000 | loss=6.7214 | elapsed=66s
  [full] step  2000/10000 | loss=6.5075 | elapsed=132s
  [full] step  3000/10000 | loss=6.2343 | elapsed=199s
  [full] step  4000/10000 | loss=6.0788 | elapsed=265s
  [full] step  5000/10000 | loss=6.1195 | elapsed=331s
  [full] step  6000/10000 | loss=5.9933 | elapsed=398s
  [full] step  7000/10000 | loss=5.9484 | elapsed=464s
  [full] step  8000/10000 | loss=5.9483 | elapsed=530s
  [full] step  9000/10000 | loss=5.9499 | elapsed=597s
  [full] ✅ Final Val PPL = 533.8243  (total 665s)
  [no_rna] step     0/10000 | loss=10.4922 | elapsed=0s
  [no_rna] step  1000/10000 | loss=6.7209 | elapsed=66s
  [no_rna] step  2000/10000 | loss=6.5053 | elapsed=132s
  [no_rna] step  3000/10000 | loss=6.2366 | elapsed=198s
  [no_rna] step  4000/10000 | loss=6.0821 | elapsed=264s
  [no_rna] step  5000/10000 | loss=6.1206 | elapsed=331s
  [no_

Everything up-to-date


## 10. T=512 Sequence Length Scaling Experiment

Tests Dense vs ADAPT-BIO at sequence length T=512 (vs T=128 in main experiments).  
The key hypothesis: ADAPT-BIO's FLOPs advantage compounds quadratically with T.

Theoretical FLOPs reduction:
- T=128, k=7 → 128/7 ≈ 18.3×
- T=512, k=7 → 512/7 ≈ 73.1×

This experiment empirically validates that the efficiency advantage
scales as expected with sequence length — the primary motivation for
sparse attention in long-sequence settings.

> **Note:** batch_size is reduced to 8 at T=512 to avoid OOM on Kaggle T4 (16 GB).

In [11]:
SEQ_LEN_512  = 512
BATCH_SIZE_512 = 8
STEPS_512    = 3000   # fewer steps — T=512 batches are ~4× more expensive

if 't512_experiment' not in results:
    print("Loading T=512 data...")
    wt2_train_512, wt2_val_512 = make_loaders(
        WikiText2Dataset, seq_len=SEQ_LEN_512, batch_size=BATCH_SIZE_512
    )
    print(f"  Train batches: {len(wt2_train_512)} | Val batches: {len(wt2_val_512)}")

    t512_results = {}

    # ── Dense @ T=512 ─────────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model_dense = FairDenseTransformer(
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN_512, dropout=0.3
    ).to(DEVICE)
    ppl_d, time_d = train_run(model_dense, wt2_train_512, wt2_val_512,
                               steps=STEPS_512, label="Dense-T512")
    t512_results['dense'] = {
        'ppl':         round(ppl_d, 4),
        'wall_time_s': round(time_d, 1),
        'flops_ratio': 1.0,
        'dense_attn_flops': SEQ_LEN_512 * SEQ_LEN_512,
    }
    del model_dense; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # ── ADAPT-BIO @ T=512 ─────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model_adapt = ADAPTBIOTransformer(
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN_512, k=7,
        anticipation_steps=100, rna_update_interval=500
    ).to(DEVICE)
    ppl_a, time_a = train_run(model_adapt, wt2_train_512, wt2_val_512,
                               steps=STEPS_512, label="ADAPT-T512")
    t512_results['adapt'] = {
        'ppl':         round(ppl_a, 4),
        'wall_time_s': round(time_a, 1),
        'flops':       theoretical_flops(7, SEQ_LEN_512),
        'flops_ratio': round(SEQ_LEN_512 / 7, 1),   # ← explicit, mirrors print block
    }
    del model_adapt; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['t512_experiment'] = t512_results
    save_results()
else:
    print("t512_experiment already done — skipping.")

# ── Print comparison table ────────────────────────────────────
r_d = results['t512_experiment']['dense']
r_a = results['t512_experiment']['adapt']

dense_attn  = SEQ_LEN_512 * SEQ_LEN_512
adapt_attn  = SEQ_LEN_512 * 7
flops_ratio = dense_attn / adapt_attn

print(f"\n  T=512 Sequence Length Experiment")
print(f"  {'Metric':<22} | {'Dense':>10} | {'ADAPT-BIO':>10}")
print("  " + "-" * 48)
print(f"  {'Val PPL ↓':<22} | {r_d['ppl']:>10.2f} | {r_a['ppl']:>10.2f}")
print(f"  {'Wall time (s) ↓':<22} | {r_d['wall_time_s']:>10.1f} | {r_a['wall_time_s']:>10.1f}")
print(f"  {'Dense attn FLOPs':<22} | {dense_attn:>10,} | {adapt_attn:>10,}")
print(f"  {'FLOPs reduction':<22} | {'1.0×':>10} | {flops_ratio:>9.1f}×")
print(f"\n  T=128 FLOPs ratio : 18.3×")
print(f"  T=512 FLOPs ratio : {flops_ratio:.1f}×  "
      f"({'%.1f' % (flops_ratio / 18.3)}× larger advantage at longer sequence)")

# ── Figure 6 ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Panel A — FLOPs ratio scaling with T
t_vals  = [64, 128, 256, 512, 1024]
ratios  = [t / 7 for t in t_vals]
axes[0].plot(t_vals, ratios, 'o-', color='#2196F3', lw=2.5, ms=8)
axes[0].axvline(x=128, color='gray', ls='--', lw=1.2, label='T=128 (main exp)')
axes[0].axvline(x=512, color='red',  ls='--', lw=1.2, label='T=512 (this exp)')
axes[0].set_xlabel('Sequence Length T', fontsize=11)
axes[0].set_ylabel('FLOPs Reduction (×) ↑', fontsize=11)
axes[0].set_title('(A) FLOPs Advantage vs T', fontweight='bold')
axes[0].legend(fontsize=9)
for t, r in zip(t_vals, ratios):
    axes[0].annotate(f'{r:.0f}×', (t, r), textcoords='offset points',
                     xytext=(5, 5), fontsize=9)

# Panel B — PPL comparison at T=512
models  = ['Dense\n(T=512)', 'ADAPT-BIO\n(T=512)']
ppls    = [r_d['ppl'], r_a['ppl']]
colors  = ['#78909C', '#4CAF50']
axes[1].bar(models, ppls, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(ppls):
    axes[1].text(i, v + 1, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Validation Perplexity ↓', fontsize=11)
axes[1].set_title('(B) Val PPL at T=512', fontweight='bold')

# Panel C — Wall time comparison
times  = [r_d['wall_time_s'], r_a['wall_time_s']]
axes[2].bar(models, times, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(times):
    axes[2].text(i, v + 1, f'{v:.0f}s', ha='center', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Wall Time (s) ↓', fontsize=11)
axes[2].set_title('(C) Training Time at T=512', fontweight='bold')

fig.suptitle(f'Figure 6: T=512 Sequence Length Scaling\n'
             f'ADAPT-BIO FLOPs advantage: 18.3× (T=128) → {flops_ratio:.1f}× (T=512)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/t512_experiment.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ Figure 6 saved → paper/figures/t512_experiment.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          f'"results: T=512 scaling experiment (mentor req 2) + Figure 6 [{flops_ratio:.0f}x FLOPs at T=512]"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")

Loading T=512 data...
  WikiText-2 [train]: 4490 chunks (seq_len=512)
  WikiText-2 [validation]: 465 chunks (seq_len=512)
  Train batches: 561 | Val batches: 58
  [Dense-T512] step     0/3000 | loss=10.5562 | elapsed=0s
  [Dense-T512] step  1000/3000 | loss=6.8480 | elapsed=72s
  [Dense-T512] step  2000/3000 | loss=6.6038 | elapsed=144s
  [Dense-T512] ✅ Final Val PPL = 773.4576  (total 216s)
  [ADAPT-T512] step     0/3000 | loss=10.4915 | elapsed=0s
  [ADAPT-T512] step  1000/3000 | loss=6.7678 | elapsed=77s
  [ADAPT-T512] step  2000/3000 | loss=6.4926 | elapsed=153s
  [ADAPT-T512] ✅ Final Val PPL = 774.2443  (total 231s)
✅ results.json saved (['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment'])

  T=512 Sequence Length Experiment
  Metric                 |      Dense |  ADAPT-BIO
  ------------------------------------------------
  Val PPL ↓              |     773.46 |     774.24
  Wall time (s) ↓        |      215.9 |   

To https://github.com/Kritika11052005/adapt-bio.git
   becadc2..1012fcb  main -> main


## 11. Real Wall-Clock Time & GPU Memory Benchmarks

Measures actual hardware performance (not theoretical FLOPs) across 4 configs:
- Dense T=128 and T=512
- ADAPT-BIO T=128 and T=512

Reports: training ms/step, inference ms/step, peak GPU memory (MB).

> These are micro-benchmarks on fixed random batches — independent of the
> training runs above, so results are reproducible without re-running full training.

In [12]:
if 'benchmarks' not in results:
    print("Running timing & memory benchmarks...")
    bench_results = {}

    BENCH_CONFIGS = [
        ('Dense-T128',   FairDenseTransformer,   128, dict(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4, num_layers=2, seq_len=128,  dropout=0.3)),
        ('ADAPT-T128',   ADAPTBIOTransformer,     128, dict(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4, num_layers=2, seq_len=128,  k=7, anticipation_steps=100, rna_update_interval=500)),
        ('Dense-T512',   FairDenseTransformer,    512, dict(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4, num_layers=2, seq_len=512,  dropout=0.3)),
        ('ADAPT-T512',   ADAPTBIOTransformer,      512, dict(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4, num_layers=2, seq_len=512,  k=7, anticipation_steps=100, rna_update_interval=500)),
    ]

    for label, ModelClass, seq_len, kwargs in BENCH_CONFIGS:
        torch.manual_seed(42)
        model = ModelClass(**kwargs).to(DEVICE)
        bench = benchmark_timing(model, seq_len=seq_len, batch_size=8,
                                 n_train_steps=50, n_infer_steps=50,
                                 label=label)
        bench['flops_ratio'] = round(seq_len / 7, 1) if 'ADAPT' in label else 1.0
        bench['seq_len']     = seq_len
        bench_results[label] = bench
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['benchmarks'] = bench_results
    save_results()
else:
    print("benchmarks already done — skipping.")

# ── Print table ───────────────────────────────────────────────
print(f"\n  {'Config':<14} | {'Train ms/step':>13} | {'Infer ms/step':>13} | {'Peak GPU MB':>11} | {'FLOPs ratio':>11}")
print("  " + "-" * 70)
for label in ['Dense-T128', 'ADAPT-T128', 'Dense-T512', 'ADAPT-T512']:
    r = results['benchmarks'][label]
    print(f"  {label:<14} | {r['train_ms_per_step']:>13.1f} | "
          f"{r['infer_ms_per_step']:>13.1f} | "
          f"{r['peak_gpu_mb']:>11.1f} | "
          f"{r['flops_ratio']:>9.1f}×")

# ── Figure 7 ─────────────────────────────────────────────────
labels_bench = ['Dense\nT=128', 'ADAPT-BIO\nT=128', 'Dense\nT=512', 'ADAPT-BIO\nT=512']
colors_bench = ['#78909C', '#4CAF50', '#455A64', '#2E7D32']

train_ms = [results['benchmarks'][k]['train_ms_per_step']  for k in ['Dense-T128','ADAPT-T128','Dense-T512','ADAPT-T512']]
infer_ms = [results['benchmarks'][k]['infer_ms_per_step']  for k in ['Dense-T128','ADAPT-T128','Dense-T512','ADAPT-T512']]
peak_mb  = [results['benchmarks'][k]['peak_gpu_mb']        for k in ['Dense-T128','ADAPT-T128','Dense-T512','ADAPT-T512']]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
x = range(len(labels_bench))
w = 0.55

# Panel A — Training time
axes[0].bar(x, train_ms, color=colors_bench, edgecolor='black', linewidth=0.8, width=w)
for i, v in enumerate(train_ms):
    axes[0].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_xticks(list(x)); axes[0].set_xticklabels(labels_bench, fontsize=9)
axes[0].set_ylabel('Training Time (ms/step) ↓', fontsize=11)
axes[0].set_title('(A) Training Time', fontweight='bold')

# Panel B — Inference time
axes[1].bar(x, infer_ms, color=colors_bench, edgecolor='black', linewidth=0.8, width=w)
for i, v in enumerate(infer_ms):
    axes[1].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_xticks(list(x)); axes[1].set_xticklabels(labels_bench, fontsize=9)
axes[1].set_ylabel('Inference Time (ms/step) ↓', fontsize=11)
axes[1].set_title('(B) Inference Time', fontweight='bold')

# Panel C — Peak GPU memory
axes[2].bar(x, peak_mb, color=colors_bench, edgecolor='black', linewidth=0.8, width=w)
for i, v in enumerate(peak_mb):
    lbl = f'{v:.0f}' if not (v != v) else 'CPU'   # nan check for CPU runs
    axes[2].text(i, v + 5, lbl, ha='center', fontsize=10, fontweight='bold')
axes[2].set_xticks(list(x)); axes[2].set_xticklabels(labels_bench, fontsize=9)
axes[2].set_ylabel('Peak GPU Memory (MB) ↓', fontsize=11)
axes[2].set_title('(C) Peak GPU Memory', fontweight='bold')

fig.suptitle('Figure 7: Real Hardware Benchmarks — Training Time, Inference Time, GPU Memory\n'
             'ADAPT-BIO vs Dense (batch_size=8, d_model=128)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/benchmarks.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 7 saved → paper/figures/benchmarks.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: real timing+memory benchmarks (mentor req 3) + Figure 7"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")

Running timing & memory benchmarks...
  [Dense-T128] train 18.1 ms/step | infer 3.9 ms/step | peak GPU 612.5 MB
  [ADAPT-T128] train 17.6 ms/step | infer 3.7 ms/step | peak GPU 620.7 MB
  [Dense-T512] train 70.8 ms/step | infer 18.8 ms/step | peak GPU 2097.3 MB
  [ADAPT-T512] train 77.6 ms/step | infer 19.5 ms/step | peak GPU 2258.7 MB
✅ results.json saved (['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks'])

  Config         | Train ms/step | Infer ms/step | Peak GPU MB | FLOPs ratio
  ----------------------------------------------------------------------
  Dense-T128     |          18.1 |           3.9 |       612.5 |       1.0×
  ADAPT-T128     |          17.6 |           3.6 |       620.7 |      18.3×
  Dense-T512     |          70.8 |          18.8 |      2097.3 |       1.0×
  ADAPT-T512     |          77.6 |          19.5 |      2258.7 |      73.1×
✅ Figure 7 saved → paper/figures/benchmarks.png
[main

To https://github.com/Kritika11052005/adapt-bio.git
   1012fcb..54ae8ea  main -> main


## 12. Aggregated Results — Tables 1 & 2

Consolidates all experiment results into the two paper tables.

In [13]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

# ── Table 1: Main results summary ────────────────────────────
print("=" * 65)
print("  TABLE 1: ADAPT-BIO vs Dense — WikiText-2 (d_model=128, 3 seeds)")
print("=" * 65)
r128 = results['main_wt2_128']
flops = r128['flops']
print(f"\n  {'Model':<16} | {'Val PPL':>8} | {'Sparsity':>9} | {'Attn FLOPs':>12} | {'Reduction':>10}")
print("  " + "-" * 65)
print(f"  {'Dense (dp=0.3)':<16} | {r128['dense']['mean']:>7.2f}±{r128['dense']['std']:.2f} | "
      f"{'0.0%':>9} | {flops['dense_attn_flops']:>12,} | {'1.0×':>10}")
print(f"  {'ADAPT-BIO':<16} | {r128['adapt']['mean']:>7.2f}±{r128['adapt']['std']:.2f} | "
      f"{'94.5%':>9} | {flops['sparse_attn_flops']:>12,} | {flops['flops_ratio']:>9.1f}×")

if 'main_wt2_256' in results:
    print(f"\n  WikiText-2 d_model=256:")
    r256 = results['main_wt2_256']
    print(f"  {'Dense (dp=0.3)':<16} | {r256['dense']['mean']:>7.2f}±{r256['dense']['std']:.2f} | "
          f"{'0.0%':>9} | {r256['flops']['dense_attn_flops']:>12,} | {'1.0×':>10}")
    print(f"  {'ADAPT-BIO':<16} | {r256['adapt']['mean']:>7.2f}±{r256['adapt']['std']:.2f} | "
          f"{'94.5%':>9} | {r256['flops']['sparse_attn_flops']:>12,} | {r256['flops']['flops_ratio']:>9.1f}×")

# ── Table 2: All ablations summary ───────────────────────────
print("\n\n" + "=" * 65)
print("  TABLE 2: Ablation & Scaling Summary")
print("=" * 65)

# ── k-sweep summary ───────────────────────────────────────────
# Read chosen_k from already-computed pareto_analysis — no hardcoding
chosen_k_wt2 = results.get('pareto_analysis', {}).get('chosen_k')

print(f"\n  k-Sweep (WikiText-2, {STEPS} steps):")
print(f"  {'k':>4} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11}")
print("  " + "-" * 40)
for kv in [int(k) for k in sorted(results['k_sweep'].keys())]:
    r = results['k_sweep'][str(kv)]
    marker = " ← chosen operating point" if kv == chosen_k_wt2 else ""
    print(f"  {kv:>4} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% "
          f"| {r['flops_ratio']:>9.1f}×{marker}")


# RNA sweep
print(f"\n  RNA Interval Sweep (WikiText-2, k=7, 10000 steps):")
frozen_ppl = results['rna_sweep']['frozen']
print(f"  {'Δ':>8} | {'Val PPL':>8} | Note")
print("  " + "-" * 45)
for delta in [50, 200, 500, 2000]:
    ppl = results['rna_sweep'][str(delta)]
    beat = "✅ beats frozen" if ppl < frozen_ppl else "❌ worse than frozen"
    print(f"  {delta:>8} | {ppl:>8.2f} | {beat}")
print(f"  {'Frozen':>8} | {frozen_ppl:>8.2f} | static mask baseline")

# Component ablation
print(f"\n  Component Ablation (WikiText-2, d_model=128):")
print(f"  {'Config':<16} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11}")
print("  " + "-" * 50)
for name, label in [('no_rna','No-RNA'), ('no_starling','No-Starling'), ('full','Full ADAPT-BIO')]:
    r = results['component_ablation'][name]
    print(f"  {label:<16} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | {r['flops_ratio']:>9.1f}×")

# T=512
print(f"\n  T=512 Scaling (WikiText-2, d_model=128):")
r_d = results['t512_experiment']['dense']
r_a = results['t512_experiment']['adapt']
ratio_512 = 512 / 7
print(f"  {'Dense T=512':<16} | PPL={r_d['ppl']:.2f} | FLOPs ratio=1.0×")
print(f"  {'ADAPT-BIO T=512':<16} | PPL={r_a['ppl']:.2f} | FLOPs ratio={ratio_512:.1f}×")
print(f"  → FLOPs advantage: 18.3× (T=128) → {ratio_512:.1f}× (T=512)")

# Benchmarks
print(f"\n  Real Hardware Benchmarks (batch_size=8):")
print(f"  {'Config':<14} | {'Train ms/step':>13} | {'Infer ms/step':>13} | {'Peak GPU MB':>11}")
print("  " + "-" * 60)
for label in ['Dense-T128', 'ADAPT-T128', 'Dense-T512', 'ADAPT-T512']:
    r = results['benchmarks'][label]
    print(f"  {label:<14} | {r['train_ms_per_step']:>13.1f} | "
          f"{r['infer_ms_per_step']:>13.1f} | {r['peak_gpu_mb']:>11.1f}")

# ── Figure 8: 2×3 Summary Panel ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Main PPL comparison d_model=128
models_main = ['Dense\n(128)', 'ADAPT\n(128)']
ppls_main   = [r128['dense']['mean'], r128['adapt']['mean']]
errs_main   = [r128['dense']['std'],  r128['adapt']['std']]
axes[0,0].bar(models_main, ppls_main, yerr=errs_main,
              color=['#78909C','#4CAF50'], edgecolor='black', linewidth=0.8,
              capsize=6, width=0.5)
for i, (v, e) in enumerate(zip(ppls_main, errs_main)):
    axes[0,0].text(i, v + e + 2, f'{v:.1f}', ha='center', fontsize=11, fontweight='bold')
axes[0,0].set_ylabel('Validation Perplexity ↓', fontsize=10)
axes[0,0].set_title('(A) Main Results — d_model=128', fontweight='bold')

# (0,1) k-sweep
K_VALS = [3, 5, 7, 9, 12]
ppls_k = [results['k_sweep'][str(k)]['ppl'] for k in K_VALS]
axes[0,1].plot(K_VALS, ppls_k, 'o-', color='#2196F3', lw=2.5, ms=8)
axes[0,1].axvline(x=7, color='red', ls='--', lw=1.5, label='k=7 operating point')
axes[0,1].set_xlabel('k (neighbours)', fontsize=10)
axes[0,1].set_ylabel('Val PPL ↓', fontsize=10)
axes[0,1].set_title('(B) k-Sweep', fontweight='bold')
axes[0,1].legend(fontsize=9)
axes[0,1].set_xticks(K_VALS)

# (0,2) RNA sweep
delta_labels = ['50','200','500','2000','Frozen']
delta_ppls   = [results['rna_sweep'][str(d)] for d in [50,200,500,2000]] + [frozen_ppl]
colors_rna   = ['#FF7043' if v >= frozen_ppl else '#4CAF50' for v in delta_ppls[:-1]] + ['#9E9E9E']
axes[0,2].bar(delta_labels, delta_ppls, color=colors_rna,
              edgecolor='black', linewidth=0.8, width=0.6)
axes[0,2].axhline(y=frozen_ppl, color='black', ls='--', lw=1.5)
axes[0,2].set_xlabel('Δ (steps)', fontsize=10)
axes[0,2].set_ylabel('Val PPL ↓', fontsize=10)
axes[0,2].set_title('(C) RNA Interval Sweep', fontweight='bold')

# (1,0) Component ablation PPL
abl_names  = ['No-RNA', 'No-Starling', 'Full']
abl_keys   = ['no_rna', 'no_starling', 'full']
abl_ppls   = [results['component_ablation'][k]['ppl'] for k in abl_keys]
abl_colors = ['#FF7043', '#FFC107', '#4CAF50']
axes[1,0].bar(abl_names, abl_ppls, color=abl_colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(abl_ppls):
    axes[1,0].text(i, v + 1, f'{v:.1f}', ha='center', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('Val PPL ↓', fontsize=10)
axes[1,0].set_title('(D) Component Ablation', fontweight='bold')

# (1,1) T=512 FLOPs scaling curve
t_range  = [64, 128, 256, 512, 1024]
ratios_t = [t / 7 for t in t_range]
axes[1,1].plot(t_range, ratios_t, 'o-', color='#2196F3', lw=2.5, ms=8)
axes[1,1].axvline(x=128, color='gray', ls='--', lw=1.2, label='T=128')
axes[1,1].axvline(x=512, color='red',  ls='--', lw=1.2, label='T=512')
axes[1,1].set_xlabel('Sequence Length T', fontsize=10)
axes[1,1].set_ylabel('FLOPs Reduction ×', fontsize=10)
axes[1,1].set_title('(E) FLOPs Scaling with T', fontweight='bold')
axes[1,1].legend(fontsize=9)
for t, r in zip(t_range, ratios_t):
    axes[1,1].annotate(f'{r:.0f}×', (t, r), textcoords='offset points',
                       xytext=(5, 5), fontsize=9)

# (1,2) Hardware benchmarks — training ms
bench_labels = ['Dense\nT=128', 'ADAPT\nT=128', 'Dense\nT=512', 'ADAPT\nT=512']
bench_keys   = ['Dense-T128', 'ADAPT-T128', 'Dense-T512', 'ADAPT-T512']
bench_colors = ['#78909C', '#4CAF50', '#455A64', '#2E7D32']
train_ms_all = [results['benchmarks'][k]['train_ms_per_step'] for k in bench_keys]
axes[1,2].bar(range(4), train_ms_all, color=bench_colors,
              edgecolor='black', linewidth=0.8, width=0.55)
axes[1,2].set_xticks(range(4))
axes[1,2].set_xticklabels(bench_labels, fontsize=9)
axes[1,2].set_ylabel('Training Time (ms/step) ↓', fontsize=10)
axes[1,2].set_title('(F) Hardware Benchmarks', fontweight='bold')
for i, v in enumerate(train_ms_all):
    axes[1,2].text(i, v + 0.3, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')

fig.suptitle('Figure 8: ADAPT-BIO — Consolidated Results Summary\n'
             'WikiText-2 | 94.5% Sparsity | 18.3× FLOPs (T=128) → 73.1× (T=512)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
os.makedirs('/kaggle/working/adapt-bio/paper/figures', exist_ok=True)
plt.savefig('/kaggle/working/adapt-bio/paper/figures/summary_panel.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 8 saved → paper/figures/summary_panel.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: aggregated tables + Figure 8 summary panel"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")

  TABLE 1: ADAPT-BIO vs Dense — WikiText-2 (d_model=128, 3 seeds)

  Model            |  Val PPL |  Sparsity |   Attn FLOPs |  Reduction
  -----------------------------------------------------------------
  Dense (dp=0.3)   |  490.57±1.92 |      0.0% |       16,384 |       1.0×
  ADAPT-BIO        |  532.98±2.06 |     94.5% |          896 |      18.3×

  WikiText-2 d_model=256:
  Dense (dp=0.3)   |  360.35±1.26 |      0.0% |       16,384 |       1.0×
  ADAPT-BIO        |  482.96±1.18 |     94.5% |          896 |      18.3×


  TABLE 2: Ablation & Scaling Summary

  k-Sweep (WikiText-2, 10000 steps):
     k |  Val PPL |  Sparsity | FLOPs ratio
  ----------------------------------------
    12 |   524.50 |     90.6% |      10.7×
     3 |   538.61 |     97.7% |      42.7× ← chosen operating point
     5 |   536.39 |     96.1% |      25.6×
     7 |   533.82 |     94.5% |      18.3×
     9 |   530.61 |     93.0% |      14.2×

  RNA Interval Sweep (WikiText-2, k=7, 10000 steps):
         Δ | 

To https://github.com/Kritika11052005/adapt-bio.git
   54ae8ea..d3f5edd  main -> main


In [14]:
# ── WikiText-2 conclusions table (generated from results) ───────────────────
r128 = results['main_wt2_128']
r256 = results['main_wt2_256']
r_d512 = results['t512_experiment']['dense']
r_a512 = results['t512_experiment']['adapt']
wt2_frozen   = results['rna_sweep']['frozen']
wt2_rna_best = min(results['rna_sweep'][str(d)] for d in [50, 200, 500, 2000])
rna_result   = "✅ beats frozen" if wt2_rna_best < wt2_frozen else "❌ scale too small"

print("=" * 60)
print("  WikiText-2 Summary of Findings")
print("=" * 60)
rows = [
    ("Main results (d=128)",
     f"Dense {r128['dense']['mean']:.2f} ± {r128['dense']['std']:.2f} "
     f"vs ADAPT {r128['adapt']['mean']:.2f} ± {r128['adapt']['std']:.2f}"),
    ("Main results (d=256)",
     f"Dense {r256['dense']['mean']:.2f} ± {r256['dense']['std']:.2f} "
     f"vs ADAPT {r256['adapt']['mean']:.2f} ± {r256['adapt']['std']:.2f}"),
    ("Sparsity (all runs)",    "94.5% — consistent across all configs"),
    ("FLOPs reduction (T=128)", "18.3×"),
    ("FLOPs reduction (T=512)", "73.1× — advantage compounds quadratically"),
    ("k=7 operating point",    "✅ chosen sparsity/quality tradeoff (k ∈ {3,5,7,9,12})"),
    ("RNA beats Frozen (d=128)", rna_result),
   ("ADAPT beats Dense T=512",
     f"{'✅' if r_a512['ppl'] < r_d512['ppl'] else '❌'} {r_a512['ppl']:.2f} vs {r_d512['ppl']:.2f}"),
]
for label, value in rows:
    print(f"  {label:<30} | {value}")

  WikiText-2 Summary of Findings
  Main results (d=128)           | Dense 490.57 ± 1.92 vs ADAPT 532.98 ± 2.06
  Main results (d=256)           | Dense 360.35 ± 1.26 vs ADAPT 482.96 ± 1.18
  Sparsity (all runs)            | 94.5% — consistent across all configs
  FLOPs reduction (T=128)        | 18.3×
  FLOPs reduction (T=512)        | 73.1× — advantage compounds quadratically
  k=7 operating point            | ✅ chosen sparsity/quality tradeoff (k ∈ {3,5,7,9,12})
  RNA beats Frozen (d=128)       | ✅ beats frozen
  ADAPT beats Dense T=512        | ❌ 774.24 vs 773.46


## 14. WikiText-103 Experiments

All experiments from Sections 6–12 are repeated on WikiText-103 to validate
that ADAPT-BIO's efficiency gains hold on a larger, more realistic dataset.


> All WT-103 results are saved under `wt103_*` keys in results.json.

In [15]:
STEPS_103 = 10000   # same as WT-2, proper training

print("Loading WikiText-103...")
wt103_train, wt103_val = make_loaders(
    WikiText103Dataset, seq_len=SEQ_LEN, batch_size=BATCH_SIZE,
    n_chunks=20000
)
print(f"  Train batches: {len(wt103_train)} | Val batches: {len(wt103_val)}")
print("WikiText-103 loaders ready ✅")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading WikiText-103...


wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  WikiText-103 [train]: 20001 chunks (seq_len=128)
  WikiText-103 [validation]: 1850 chunks (seq_len=128)
  Train batches: 625 | Val batches: 57
WikiText-103 loaders ready ✅


## 15. Main Results — Dense vs ADAPT-BIO (WikiText-103)

3 seeds × 2 model sizes (d_model=128, d_model=256) on WikiText-103.
Same protocol as Section 6 (WikiText-2), STEPS_103=10000 steps per run.
Results saved under `main_wt103_128` / `main_wt103_256` — resumable per size.

In [16]:

# ── d_model = 128  (WikiText-103) ───────────────────────────────────────────
if 'main_wt103_128' not in results:
    print("=" * 55)
    print("  WikiText-103 | d_model = 128")
    print("=" * 55)
 
    dense_ppls = run_seeds(
        FairDenseTransformer, "Dense-103-128", SEEDS,
        loader_train=wt103_train, loader_val=wt103_val,   # ← WT-103 loaders
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN)
 
    adapt_ppls = run_seeds(
        ADAPTBIOTransformer, "ADAPT-103-128", SEEDS,
        loader_train=wt103_train, loader_val=wt103_val,   # ← WT-103 loaders
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=500)
 
    results['main_wt103_128'] = {
        'dense': {'ppls': dense_ppls,
                  'mean': round(float(np.mean(dense_ppls)), 4),
                  'std':  round(float(np.std(dense_ppls)),  4)},
        'adapt': {'ppls': adapt_ppls,
                  'mean': round(float(np.mean(adapt_ppls)), 4),
                  'std':  round(float(np.std(adapt_ppls)),  4)},
        'flops': theoretical_flops(7, SEQ_LEN),
    }
    save_results()
else:
    print("main_wt103_128 already done — skipping.")
 
r = results['main_wt103_128']
print(f"\n  Dense-103-128 : PPL = {r['dense']['mean']:.2f} ± {r['dense']['std']:.2f}")
print(f"  ADAPT-103-128 : PPL = {r['adapt']['mean']:.2f} ± {r['adapt']['std']:.2f}")
 
 
# ── d_model = 256  (WikiText-103) ───────────────────────────────────────────
if 'main_wt103_256' not in results:
    print("\n" + "=" * 55)
    print("  WikiText-103 | d_model = 256")
    print("=" * 55)
 
    dense_ppls = run_seeds(
        FairDenseTransformer, "Dense-103-256", SEEDS,
        loader_train=wt103_train, loader_val=wt103_val,   # ← WT-103 loaders
        vocab_size=VOCAB_SIZE, d_model=256, num_heads=8,
        num_layers=2, seq_len=SEQ_LEN)
 
    adapt_ppls = run_seeds(
        ADAPTBIOTransformer, "ADAPT-103-256", SEEDS,
        loader_train=wt103_train, loader_val=wt103_val,   # ← WT-103 loaders
        vocab_size=VOCAB_SIZE, d_model=256, num_heads=8,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=500)
 
    results['main_wt103_256'] = {
        'dense': {'ppls': dense_ppls,
                  'mean': round(float(np.mean(dense_ppls)), 4),
                  'std':  round(float(np.std(dense_ppls)),  4)},
        'adapt': {'ppls': adapt_ppls,
                  'mean': round(float(np.mean(adapt_ppls)), 4),
                  'std':  round(float(np.std(adapt_ppls)),  4)},
        'flops': theoretical_flops(7, SEQ_LEN),
    }
    save_results()
else:
    print("main_wt103_256 already done — skipping.")
 
r2 = results['main_wt103_256']
print(f"\n  Dense-103-256 : PPL = {r2['dense']['mean']:.2f} ± {r2['dense']['std']:.2f}")
print(f"  ADAPT-103-256 : PPL = {r2['adapt']['mean']:.2f} ± {r2['adapt']['std']:.2f}")
 
# ── Commit WikiText-103 main results ──────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: WikiText-103 main results d=128 + d=256 '
          '(3 seeds, correct loaders, EMA-fixed)"')
os.system(f"cd /kaggle/working/adapt-bio && git push "
          f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main")
print("✅ Pushed to GitHub")
 

  WikiText-103 | d_model = 128
  [Dense-103-128-s42] step     0/10000 | loss=10.5533 | elapsed=0s
  [Dense-103-128-s42] step  1000/10000 | loss=6.7672 | elapsed=71s
  [Dense-103-128-s42] step  2000/10000 | loss=6.7467 | elapsed=141s
  [Dense-103-128-s42] step  3000/10000 | loss=6.4225 | elapsed=210s
  [Dense-103-128-s42] step  4000/10000 | loss=6.4238 | elapsed=279s
  [Dense-103-128-s42] step  5000/10000 | loss=6.1856 | elapsed=349s
  [Dense-103-128-s42] step  6000/10000 | loss=6.1126 | elapsed=418s
  [Dense-103-128-s42] step  7000/10000 | loss=6.2102 | elapsed=487s
  [Dense-103-128-s42] step  8000/10000 | loss=6.0684 | elapsed=556s
  [Dense-103-128-s42] step  9000/10000 | loss=6.1591 | elapsed=626s
  [Dense-103-128-s42] ✅ Final Val PPL = 485.4258  (total 697s)
  [Dense-103-128-s123] step     0/10000 | loss=10.5463 | elapsed=0s
  [Dense-103-128-s123] step  1000/10000 | loss=6.7784 | elapsed=69s
  [Dense-103-128-s123] step  2000/10000 | loss=6.6336 | elapsed=139s
  [Dense-103-128-s123] 

Everything up-to-date


In [17]:
from scipy import stats

print("=" * 70)
print("  STATISTICAL SIGNIFICANCE — Paired t-test + Effect Size (ADAPT vs Dense)")
print("  H0: mean(ADAPT PPL) == mean(Dense PPL) per config")
print()
print("  ⚠️  LIMITATION: 3 seeds → df=2 for t-test. With df=2, the critical")
print("  t-value for α=0.05 (two-tailed) is 4.303 — an extremely high bar.")
print("  We therefore report Cohen's d (effect size) alongside p-values,")
print("  and interpret |d| > 0.8 as 'large effect' regardless of p-value.")
print("  Journal reviewers should note this underpowered test; 5+ seeds")
print("  are recommended for future work.")
print("=" * 70)

sig_configs = [
    ("WT-2  d=128",  "main_wt2_128"),
    ("WT-2  d=256",  "main_wt2_256"),
    ("WT-103 d=128", "main_wt103_128"),
    ("WT-103 d=256", "main_wt103_256"),
]

sig_results = {}

for label, key in sig_configs:
    r = results[key]
    dense_ppls = r['dense']['ppls']
    adapt_ppls = r['adapt']['ppls']

    t_stat, p_val = stats.ttest_rel(adapt_ppls, dense_ppls)

    # Cohen's d for paired samples
    diffs    = [a - d for a, d in zip(adapt_ppls, dense_ppls)]
    d_mean   = float(np.mean(diffs))
    d_std    = float(np.std(diffs, ddof=1))
    cohens_d = d_mean / d_std if d_std > 0 else float('inf')

    gap_pct = (r['adapt']['mean'] - r['dense']['mean']) / r['dense']['mean'] * 100
    sig_str = "✅ p<0.05" if p_val < 0.05 else "⚠️  p≥0.05 (df=2, low power)"

    print(f"\n  {label}:")
    print(f"    Dense  : {r['dense']['mean']:.4f} ± {r['dense']['std']:.4f}")
    print(f"    ADAPT  : {r['adapt']['mean']:.4f} ± {r['adapt']['std']:.4f}")
    print(f"    Gap    : {gap_pct:+.1f}%")
    print(f"    t={t_stat:.3f}  p={p_val:.4f}  {sig_str}")
    print(f"    Cohen's d = {cohens_d:.3f}  "
          f"({'large' if abs(cohens_d) > 0.8 else 'medium' if abs(cohens_d) > 0.5 else 'small'} effect)")

    sig_results[key] = {
        't_stat':   round(t_stat, 4),
        'p_val':    round(p_val, 4),
        'cohens_d': round(cohens_d, 4),
        'gap_pct':  round(gap_pct, 2),
    }

results['significance_tests'] = sig_results
save_results()
print("\n✅ Significance tests (with Cohen's d) saved to results.json")


  STATISTICAL SIGNIFICANCE — Paired t-test + Effect Size (ADAPT vs Dense)
  H0: mean(ADAPT PPL) == mean(Dense PPL) per config

  ⚠️  LIMITATION: 3 seeds → df=2 for t-test. With df=2, the critical
  t-value for α=0.05 (two-tailed) is 4.303 — an extremely high bar.
  We therefore report Cohen's d (effect size) alongside p-values,
  and interpret |d| > 0.8 as 'large effect' regardless of p-value.
  Journal reviewers should note this underpowered test; 5+ seeds
  are recommended for future work.

  WT-2  d=128:
    Dense  : 490.5667 ± 1.9193
    ADAPT  : 532.9822 ± 2.0640
    Gap    : +8.6%
    t=15.513  p=0.0041  ✅ p<0.05
    Cohen's d = 8.957  (large effect)

  WT-2  d=256:
    Dense  : 360.3462 ± 1.2573
    ADAPT  : 482.9559 ± 1.1802
    Gap    : +34.0%
    t=139.422  p=0.0001  ✅ p<0.05
    Cohen's d = 80.496  (large effect)

  WT-103 d=128:
    Dense  : 485.5922 ± 1.5781
    ADAPT  : 523.7804 ± 1.7825
    Gap    : +7.9%
    t=16.081  p=0.0038  ✅ p<0.05
    Cohen's d = 9.284  (large eff

## 16. k-Sweep — WikiText-103

Same protocol as Section 7 (WikiText-2 k-sweep), now on WikiText-103.
5 values of k, 1 seed, 10000 steps. Results saved under `wt103_k_sweep`.

In [18]:
K_VALS = [3, 5, 7, 9, 12]

if 'wt103_k_sweep' not in results:
    print("Running WikiText-103 k-sweep (5 runs × 10000 steps)...")
    k103_results = {}
    for kv in K_VALS:
        torch.manual_seed(42); np.random.seed(42)
        model = ADAPTBIOTransformer(
            vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
            num_layers=2, seq_len=SEQ_LEN, k=kv,
            anticipation_steps=100, rna_update_interval=500
        ).to(DEVICE)
        ppl, _ = train_run(model, wt103_train, wt103_val,
                           steps=STEPS_103, label=f"WT103-k={kv}")
        flops = theoretical_flops(kv, SEQ_LEN)
        k103_results[str(kv)] = {
            'ppl':          round(ppl, 4),
            'sparsity_pct': round(flops['flop_reduction'] * 100, 2),
            'flops_ratio':  round(flops['flops_ratio'], 2),
        }
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    results['wt103_k_sweep'] = k103_results
    save_results()
else:
    print("wt103_k_sweep already done — skipping.")

# ── Find Pareto-optimal k dynamically ────────────────────────
# is_pareto_optimal already defined in Section 7 with signature
# (k_candidate, sw, k_vals) — do NOT redefine here, just call it.

sweep    = results['wt103_k_sweep']
K_VALS_P = sorted(int(k) for k in sweep.keys())

pareto_ks = [k for k in K_VALS_P if is_pareto_optimal(k, sweep, K_VALS_P)]
optimal_k = max(pareto_ks, key=lambda k: sweep[str(k)]['sparsity_pct'])

print(f"  Pareto-optimal k values: {pareto_ks}")
print(f"  Chosen operating point : k={optimal_k} "
      f"(highest sparsity among non-dominated)")

# ── Print table ───────────────────────────────────────────────
print(f"\n  {'k':>4} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11} | {'Status':>24}")
print("  " + "-"*60)
for kv in K_VALS_P:
    r = sweep[str(kv)]
    if kv == optimal_k:
        status = "← chosen operating point"
    elif kv in pareto_ks:      # ← colon fixed
        status = "Pareto-optimal"
    else:
        status = "dominated"
    print(f"  {kv:>4} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | "
          f"{r['flops_ratio']:>9.1f}× | {status:>24}")


# ── Figure ────────────────────────────────────────────────────
ppls = [results['wt103_k_sweep'][str(k)]['ppl'] for k in K_VALS]
spar = [results['wt103_k_sweep'][str(k)]['sparsity_pct'] for k in K_VALS]

fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()
ax1.plot(K_VALS, ppls, 'o-', color='#2196F3', lw=2.5, ms=8, label='Val PPL ↓')
ax1.axvline(x=7, color='red', ls='--', lw=1.5, label='k=7 (chosen operating point)')
ax2.plot(K_VALS, spar, 's--', color='#4CAF50', lw=2, ms=8, label='Sparsity %')
ax1.set_xlabel('k (neighbours per token)', fontsize=12)
ax1.set_ylabel('Validation Perplexity ↓', color='#2196F3', fontsize=12)
ax2.set_ylabel('Attention Sparsity % ↑', color='#4CAF50', fontsize=12)
ax2.set_ylim(85, 100)
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc='upper right', fontsize=10)
plt.title('Figure 9: k-Sweep — WikiText-103\n'
          'ADAPT-BIO (10000 steps, d_model=128)',
          fontsize=12, fontweight='bold')
plt.xticks(K_VALS)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/wt103_k_sweep.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 9 saved → paper/figures/wt103_k_sweep.png")

# ── Pareto frontier analysis — WikiText-103 ───────────────────
# is_pareto_optimal already defined in Section 7 — do NOT redefine
sweep    = results['wt103_k_sweep']
K_VALS_P = sorted(int(k) for k in sweep.keys())

pareto_ks   = [k for k in K_VALS_P if is_pareto_optimal(k, sweep, K_VALS_P)]
chosen_k    = max(pareto_ks, key=lambda k: sweep[str(k)]['sparsity_pct'])
chosen_ppl  = sweep[str(chosen_k)]['ppl']
chosen_spar = sweep[str(chosen_k)]['sparsity_pct']

print("=" * 60)
print("  PARETO FRONTIER ANALYSIS — WikiText-103 k-sweep")
print("=" * 60)
print(f"\n  Pareto-optimal k values : {pareto_ks}")
print(f"  Chosen operating point  : k={chosen_k} "
      f"(PPL={chosen_ppl:.2f}, Sparsity={chosen_spar:.1f}%)\n")

print(f"  {'k':>4} | {'PPL':>8} | {'Sparsity':>9} | {'Status':>25}")
print("  " + "-" * 52)
for kv in K_VALS_P:
    r = sweep[str(kv)]
    if kv == chosen_k:
        status = "← chosen operating point"
    elif kv in pareto_ks:
        status = "Pareto-optimal"
    else:
        status = "dominated"
    print(f"  {kv:>4} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | {status:>25}")

dominated_by = [
    kv for kv in K_VALS_P if kv != chosen_k
    and sweep[str(kv)]['ppl'] < chosen_ppl
    and sweep[str(kv)]['sparsity_pct'] >= chosen_spar
]

if dominated_by:
    print(f"\n  ⚠️  k={dominated_by} dominates chosen k={chosen_k} "
          f"— operating point needs revision.")
else:
    print(f"\n  ✅ k={chosen_k} is Pareto-optimal on WikiText-103 — "
          f"no k achieves both lower PPL and equal or higher sparsity.")

# Cross-check against WT-2 result
wt2_chosen_k = results.get('pareto_analysis', {}).get('chosen_k')
if wt2_chosen_k is not None:
    if chosen_k == wt2_chosen_k:
        print(f"\n  ✅ WT-103 chosen k={chosen_k} agrees with "
              f"WT-2 chosen k={wt2_chosen_k} — dataset-independent.")
    else:
        print(f"\n  ⚠️  WT-103 chosen k={chosen_k} differs from "
              f"WT-2 chosen k={wt2_chosen_k} — investigate before "
              f"finalising paper claim.")

results['wt103_pareto_analysis'] = {
    'pareto_ks':      pareto_ks,
    'chosen_k':       chosen_k,
    'chosen_ppl':     chosen_ppl,
    'chosen_spar':    chosen_spar,
    'dominated_by':   dominated_by,
    'pareto_optimal': len(dominated_by) == 0,
    'agrees_with_wt2': chosen_k == wt2_chosen_k if wt2_chosen_k else None,
}
save_results()
print("\n✅ WT-103 Pareto analysis saved to results.json")
# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: WikiText-103 k-sweep + Figure 9"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

Running WikiText-103 k-sweep (5 runs × 10000 steps)...
  [WT103-k=3] step     0/10000 | loss=10.4895 | elapsed=0s
  [WT103-k=3] step  1000/10000 | loss=6.8154 | elapsed=66s
  [WT103-k=3] step  2000/10000 | loss=6.5862 | elapsed=132s
  [WT103-k=3] step  3000/10000 | loss=6.3863 | elapsed=199s
  [WT103-k=3] step  4000/10000 | loss=6.3429 | elapsed=266s
  [WT103-k=3] step  5000/10000 | loss=6.0338 | elapsed=332s
  [WT103-k=3] step  6000/10000 | loss=5.9735 | elapsed=398s
  [WT103-k=3] step  7000/10000 | loss=6.1534 | elapsed=465s
  [WT103-k=3] step  8000/10000 | loss=5.9548 | elapsed=531s
  [WT103-k=3] step  9000/10000 | loss=5.9811 | elapsed=597s
  [WT103-k=3] ✅ Final Val PPL = 528.9995  (total 666s)
  [WT103-k=5] step     0/10000 | loss=10.4895 | elapsed=0s
  [WT103-k=5] step  1000/10000 | loss=6.8164 | elapsed=66s
  [WT103-k=5] step  2000/10000 | loss=6.5854 | elapsed=133s
  [WT103-k=5] step  3000/10000 | loss=6.3835 | elapsed=199s
  [WT103-k=5] step  4000/10000 | loss=6.3367 | elapsed

## 17. RNA Interval Sweep — WikiText-103

Same protocol as Section 8, now on WikiText-103.
The critical comparison: Δ=500 vs Frozen — validates dynamic revision on the larger dataset.
Results saved under `wt103_rna_sweep`.

In [19]:
DELTA_VALS = [50, 200, 500, 2000]

if 'wt103_rna_sweep' not in results:
    print("Running WikiText-103 RNA sweep (5 runs × 10000 steps)...")
    rna103_results = {}

    for delta in DELTA_VALS:
        torch.manual_seed(42); np.random.seed(42)
        model = ADAPTBIOTransformer(
            vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
            num_layers=2, seq_len=SEQ_LEN, k=7,
            anticipation_steps=100, rna_update_interval=delta
        ).to(DEVICE)
        ppl, _ = train_run(model, wt103_train, wt103_val,
                           steps=STEPS_103, label=f"WT103-Δ={delta}")
        rna103_results[str(delta)] = round(ppl, 4)
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # ── Frozen baseline ───────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model = ADAPTBIOTransformer(
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN, k=7,
        anticipation_steps=100, rna_update_interval=999999,
        use_rna=False
    ).to(DEVICE)
    ppl, _ = train_run(model, wt103_train, wt103_val,
                       steps=STEPS_103, label="WT103-Frozen")
    rna103_results['frozen'] = round(ppl, 4)
    del model; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['wt103_rna_sweep'] = rna103_results
    save_results()
else:
    print("wt103_rna_sweep already done — skipping.")

# ── Print table ───────────────────────────────────────────────
frozen103 = results['wt103_rna_sweep']['frozen']
print(f"\n  {'Δ (steps)':>12} | {'Val PPL':>8} | Note")
print("  " + "-" * 45)
for delta in DELTA_VALS:
    ppl = results['wt103_rna_sweep'][str(delta)]
    beat = "✅ beats frozen" if ppl < frozen103 else "❌ worse than frozen"
    print(f"  {delta:>12} | {ppl:>8.2f} | {beat}")
print(f"  {'Frozen':>12} | {frozen103:>8.2f} | static mask baseline")

# ── Figure ────────────────────────────────────────────────────
x_labels = [str(d) for d in DELTA_VALS] + ["Frozen"]
y_vals    = [results['wt103_rna_sweep'][str(d)] for d in DELTA_VALS] + [frozen103]
colors    = ['#FF7043' if v >= frozen103 else '#4CAF50' for v in y_vals[:-1]] + ['#9E9E9E']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(x_labels, y_vals, color=colors, width=0.6,
              edgecolor='black', linewidth=0.8)
ax.axhline(y=frozen103, color='black', ls='--', lw=1.5,
           label=f'Frozen baseline (PPL={frozen103:.2f})')
for bar, val in zip(bars, y_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
            f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xlabel('RNA Update Interval Δ (steps)', fontsize=12)
ax.set_ylabel('Validation Perplexity ↓', fontsize=12)
ax.legend(fontsize=10)
plt.title('Figure 10: RNA Interval Sweep — WikiText-103\n'
          'ADAPT-BIO (k=7, 10000 steps, d_model=128)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/wt103_rna_sweep.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 10 saved → paper/figures/wt103_rna_sweep.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: WikiText-103 RNA sweep + Figure 10"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

Running WikiText-103 RNA sweep (5 runs × 10000 steps)...
  [WT103-Δ=50] step     0/10000 | loss=10.4895 | elapsed=0s
  [WT103-Δ=50] step  1000/10000 | loss=6.8133 | elapsed=66s
  [WT103-Δ=50] step  2000/10000 | loss=6.5842 | elapsed=133s
  [WT103-Δ=50] step  3000/10000 | loss=6.3904 | elapsed=199s
  [WT103-Δ=50] step  4000/10000 | loss=6.3442 | elapsed=265s
  [WT103-Δ=50] step  5000/10000 | loss=6.0342 | elapsed=332s
  [WT103-Δ=50] step  6000/10000 | loss=5.9690 | elapsed=399s
  [WT103-Δ=50] step  7000/10000 | loss=6.1549 | elapsed=465s
  [WT103-Δ=50] step  8000/10000 | loss=5.9539 | elapsed=531s
  [WT103-Δ=50] step  9000/10000 | loss=5.9715 | elapsed=598s
  [WT103-Δ=50] ✅ Final Val PPL = 525.6540  (total 665s)
  [WT103-Δ=200] step     0/10000 | loss=10.4895 | elapsed=0s
  [WT103-Δ=200] step  1000/10000 | loss=6.8192 | elapsed=66s
  [WT103-Δ=200] step  2000/10000 | loss=6.5838 | elapsed=131s
  [WT103-Δ=200] step  3000/10000 | loss=6.3839 | elapsed=197s
  [WT103-Δ=200] step  4000/10000 

## 18. Component Ablation — WikiText-103

Same protocol as Section 9 (WT-2 component ablation), now on WikiText-103.
Full ADAPT-BIO vs No-RNA vs No-Starling. Results saved under `wt103_component_ablation`.

In [20]:
ABLATION_CONFIGS = {
    'full':          dict(use_rna=True,  use_starling=True),
    'no_rna':        dict(use_rna=False, use_starling=True),
    'no_starling':   dict(use_rna=True,  use_starling=False),
    'movement_only': dict(use_rna=False, use_starling=False),
}

if 'wt103_component_ablation' not in results:
    print(f"Running WikiText-103 component ablation ({len(ABLATION_CONFIGS)} configs × {STEPS_103} steps)...")
    abl103_results = {}

    for name, flags in ABLATION_CONFIGS.items():
        torch.manual_seed(42); np.random.seed(42)
        k_val = 7 if flags['use_starling'] else SEQ_LEN

        model = ADAPTBIOTransformer(
            vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
            num_layers=2, seq_len=SEQ_LEN, k=k_val,
            anticipation_steps=100, rna_update_interval=500,
            **flags
        ).to(DEVICE)
        ppl, wall_time = train_run(model, wt103_train, wt103_val,
                                   steps=STEPS_103, label=f"WT103-{name}")
        flops = theoretical_flops(k_val, SEQ_LEN)
        abl103_results[name] = {
            'ppl':          round(ppl, 4),
            'sparsity_pct': round(flops['flop_reduction'] * 100, 2),
            'flops_ratio':  round(flops['flops_ratio'], 2),
            'wall_time_s':  round(wall_time, 1),
        }
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['wt103_component_ablation'] = abl103_results
    save_results()
else:
    print("wt103_component_ablation already done — skipping.")

# ── Print table ───────────────────────────────────────────────
print(f"\n  {'Config':<16} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11} | {'Time (s)':>9}")
print("  " + "-" * 62)
for name in ['no_rna', 'no_starling', 'full']:
    r = results['wt103_component_ablation'][name]
    label = {'full': 'Full ADAPT-BIO', 'no_rna': 'No-RNA',
             'no_starling': 'No-Starling'}[name]
    print(f"  {label:<16} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% "
          f"| {r['flops_ratio']:>9.1f}× | {r['wall_time_s']:>9.1f}")

# ── Figure 11 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
configs    = ['no_rna', 'no_starling', 'full']
labels     = ['No-RNA\n(frozen mask)', 'No-Starling\n(dense topology)', 'Full\nADAPT-BIO']
bar_colors = ['#FF7043', '#FFC107', '#4CAF50']

ppls  = [results['wt103_component_ablation'][c]['ppl']          for c in configs]
spars = [results['wt103_component_ablation'][c]['sparsity_pct'] for c in configs]
flops = [results['wt103_component_ablation'][c]['flops_ratio']  for c in configs]

axes[0].bar(labels, ppls, color=bar_colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(ppls):
    axes[0].text(i, v + 1, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Validation Perplexity ↓', fontsize=11)
axes[0].set_title('(A) PPL by Component', fontweight='bold')

axes[1].bar(labels, spars, color=bar_colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(spars):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, fontweight='bold')
axes[1].set_ylabel('Attention Sparsity % ↑', fontsize=11)
axes[1].set_ylim(0, 105)
axes[1].set_title('(B) Sparsity by Component', fontweight='bold')

axes[2].bar(labels, flops, color=bar_colors, edgecolor='black', linewidth=0.8)
for i, v in enumerate(flops):
    axes[2].text(i, v + 0.2, f'{v:.1f}×', ha='center', fontsize=10, fontweight='bold')
axes[2].set_ylabel('FLOPs Reduction (×) ↑', fontsize=11)
axes[2].set_title('(C) FLOPs Ratio by Component', fontweight='bold')

fig.suptitle('Figure 11: SOMA Component Ablation — WikiText-103\n'
             'ADAPT-BIO (d_model=128, 10000 steps)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/wt103_component_ablation.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 11 saved → paper/figures/wt103_component_ablation.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: WikiText-103 component ablation + Figure 11"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

Running WikiText-103 component ablation (4 configs × 10000 steps)...
  [WT103-full] step     0/10000 | loss=10.4895 | elapsed=0s
  [WT103-full] step  1000/10000 | loss=6.8174 | elapsed=66s
  [WT103-full] step  2000/10000 | loss=6.5814 | elapsed=133s
  [WT103-full] step  3000/10000 | loss=6.3800 | elapsed=200s
  [WT103-full] step  4000/10000 | loss=6.3354 | elapsed=266s
  [WT103-full] step  5000/10000 | loss=6.0254 | elapsed=332s
  [WT103-full] step  6000/10000 | loss=5.9706 | elapsed=399s
  [WT103-full] step  7000/10000 | loss=6.1474 | elapsed=465s
  [WT103-full] step  8000/10000 | loss=5.9461 | elapsed=532s
  [WT103-full] step  9000/10000 | loss=5.9647 | elapsed=599s
  [WT103-full] ✅ Final Val PPL = 524.1496  (total 667s)
  [WT103-no_rna] step     0/10000 | loss=10.4895 | elapsed=0s
  [WT103-no_rna] step  1000/10000 | loss=6.8138 | elapsed=66s
  [WT103-no_rna] step  2000/10000 | loss=6.5844 | elapsed=131s
  [WT103-no_rna] step  3000/10000 | loss=6.3823 | elapsed=197s
  [WT103-no_rna] 

## 19. T=512 Scaling Experiment — WikiText-103

Same protocol as Section 10 (WT-2 T=512), now on WikiText-103.
Dense vs ADAPT-BIO at sequence length T=512.
Results saved under `wt103_t512_experiment`.

In [21]:
SEQ_LEN_512    = 512
BATCH_SIZE_512 = 8
STEPS_512_103  = 3000   # T=512 batches are ~4× more expensive

if 'wt103_t512_experiment' not in results:
    print("Loading WikiText-103 T=512 data...")
    wt103_train_512, wt103_val_512 = make_loaders(
        WikiText103Dataset, seq_len=SEQ_LEN_512,
        batch_size=BATCH_SIZE_512, n_chunks=20000
    )
    print(f"  Train batches: {len(wt103_train_512)} | Val batches: {len(wt103_val_512)}")

    t512_103_results = {}

    # ── Dense @ T=512 ─────────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model_dense = FairDenseTransformer(
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN_512, dropout=0.3
    ).to(DEVICE)
    ppl_d, time_d = train_run(model_dense, wt103_train_512, wt103_val_512,
                               steps=STEPS_512_103, label="WT103-Dense-T512")
    t512_103_results['dense'] = {
        'ppl':         round(ppl_d, 4),
        'wall_time_s': round(time_d, 1),
        'flops_ratio': 1.0,
        'dense_attn_flops': SEQ_LEN_512 * SEQ_LEN_512,
    }
    del model_dense; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    # ── ADAPT-BIO @ T=512 ─────────────────────────────────────
    torch.manual_seed(42); np.random.seed(42)
    model_adapt = ADAPTBIOTransformer(
        vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
        num_layers=2, seq_len=SEQ_LEN_512, k=7,
        anticipation_steps=100, rna_update_interval=500
    ).to(DEVICE)
    ppl_a, time_a = train_run(model_adapt, wt103_train_512, wt103_val_512,
                               steps=STEPS_512_103, label="WT103-ADAPT-T512")
    t512_103_results['adapt'] = {
        'ppl':         round(ppl_a, 4),
        'wall_time_s': round(time_a, 1),
        'flops':       theoretical_flops(7, SEQ_LEN_512),
        'flops_ratio': round(SEQ_LEN_512 / 7, 1),
    }
    del model_adapt; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['wt103_t512_experiment'] = t512_103_results
    save_results()
else:
    print("wt103_t512_experiment already done — skipping.")

# ── Print comparison ──────────────────────────────────────────
r_d = results['wt103_t512_experiment']['dense']
r_a = results['wt103_t512_experiment']['adapt']
flops_ratio_512 = SEQ_LEN_512 / 7

print(f"\n  WikiText-103 | T=512 Experiment")
print(f"  {'Metric':<22} | {'Dense':>10} | {'ADAPT-BIO':>10}")
print("  " + "-" * 48)
print(f"  {'Val PPL ↓':<22} | {r_d['ppl']:>10.2f} | {r_a['ppl']:>10.2f}")
print(f"  {'Wall time (s) ↓':<22} | {r_d['wall_time_s']:>10.1f} | {r_a['wall_time_s']:>10.1f}")
print(f"  {'FLOPs reduction':<22} | {'1.0×':>10} | {flops_ratio_512:>9.1f}×")
print(f"\n  T=128 FLOPs ratio : 18.3×")
print(f"  T=512 FLOPs ratio : {flops_ratio_512:.1f}×")

# ── Figure 12 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

models  = ['Dense\n(T=512)', 'ADAPT-BIO\n(T=512)']
ppls    = [r_d['ppl'], r_a['ppl']]
times   = [r_d['wall_time_s'], r_a['wall_time_s']]
colors  = ['#78909C', '#4CAF50']

axes[0].bar(models, ppls, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(ppls):
    axes[0].text(i, v + 1, f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Validation Perplexity ↓', fontsize=11)
axes[0].set_title('(A) Val PPL at T=512', fontweight='bold')

axes[1].bar(models, times, color=colors, edgecolor='black', linewidth=0.8, width=0.5)
for i, v in enumerate(times):
    axes[1].text(i, v + 1, f'{v:.0f}s', ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Wall Time (s) ↓', fontsize=11)
axes[1].set_title('(B) Training Time at T=512', fontweight='bold')

fig.suptitle(f'Figure 12: T=512 Scaling — WikiText-103\n'
             f'FLOPs advantage: 18.3× (T=128) → {flops_ratio_512:.1f}× (T=512)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/wt103_t512_experiment.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 12 saved → paper/figures/wt103_t512_experiment.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: WikiText-103 T=512 scaling + Figure 12"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading WikiText-103 T=512 data...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  WikiText-103 [train]: 20000 chunks (seq_len=512)
  WikiText-103 [validation]: 465 chunks (seq_len=512)
  Train batches: 2500 | Val batches: 58
  [WT103-Dense-T512] step     0/3000 | loss=10.5927 | elapsed=0s
  [WT103-Dense-T512] step  1000/3000 | loss=6.8472 | elapsed=73s
  [WT103-Dense-T512] step  2000/3000 | loss=6.6983 | elapsed=145s
  [WT103-Dense-T512] ✅ Final Val PPL = 758.5070  (total 217s)
  [WT103-ADAPT-T512] step     0/3000 | loss=10.5115 | elapsed=0s
  [WT103-ADAPT-T512] step  1000/3000 | loss=6.8669 | elapsed=77s
  [WT103-ADAPT-T512] step  2000/3000 | loss=6.4967 | elapsed=153s
  [WT103-ADAPT-T512] ✅ Final Val PPL = 754.0157  (total 230s)
✅ results.json saved (['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_experiment'])

  WikiText-1

## 20. Hardware Benchmarks — WikiText-103

Real wall-clock timing and GPU memory across 4 configs on WikiText-103 data.
Results saved under `wt103_benchmarks`.

In [22]:
if 'wt103_benchmarks' not in results:
    print("Running WikiText-103 hardware benchmarks...")
    bench103_results = {}

    BENCH_CONFIGS_103 = [
        ('WT103-Dense-T128',  FairDenseTransformer, 128,
         dict(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
              num_layers=2, seq_len=128, dropout=0.3)),
        ('WT103-ADAPT-T128',  ADAPTBIOTransformer,  128,
         dict(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
              num_layers=2, seq_len=128, k=7,
              anticipation_steps=100, rna_update_interval=500)),
        ('WT103-Dense-T512',  FairDenseTransformer, 512,
         dict(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
              num_layers=2, seq_len=512, dropout=0.3)),
        ('WT103-ADAPT-T512',  ADAPTBIOTransformer,  512,
         dict(vocab_size=VOCAB_SIZE, d_model=128, num_heads=4,
              num_layers=2, seq_len=512, k=7,
              anticipation_steps=100, rna_update_interval=500)),
    ]

    for label, ModelClass, seq_len, kwargs in BENCH_CONFIGS_103:
        torch.manual_seed(42)
        model = ModelClass(**kwargs).to(DEVICE)
        bench = benchmark_timing(model, seq_len=seq_len, batch_size=8,
                                 n_train_steps=50, n_infer_steps=50,
                                 label=label)
        bench['flops_ratio'] = round(seq_len / 7, 1) if 'ADAPT' in label else 1.0
        bench['seq_len']     = seq_len
        bench103_results[label] = bench
        del model; gc.collect()
        if DEVICE.type == "cuda": torch.cuda.empty_cache()

    results['wt103_benchmarks'] = bench103_results
    save_results()
else:
    print("wt103_benchmarks already done — skipping.")

# ── Print table ───────────────────────────────────────────────
print(f"\n  {'Config':<18} | {'Train ms/step':>13} | {'Infer ms/step':>13} | {'Peak GPU MB':>11} | {'FLOPs ratio':>11}")
print("  " + "-" * 75)
for label in ['WT103-Dense-T128', 'WT103-ADAPT-T128', 'WT103-Dense-T512', 'WT103-ADAPT-T512']:
    r = results['wt103_benchmarks'][label]
    print(f"  {label:<18} | {r['train_ms_per_step']:>13.1f} | "
          f"{r['infer_ms_per_step']:>13.1f} | "
          f"{r['peak_gpu_mb']:>11.1f} | "
          f"{r['flops_ratio']:>9.1f}×")

# ── Figure 13 ────────────────────────────────────────────────
labels_b = ['Dense\nT=128', 'ADAPT\nT=128', 'Dense\nT=512', 'ADAPT\nT=512']
keys_b   = ['WT103-Dense-T128', 'WT103-ADAPT-T128', 'WT103-Dense-T512', 'WT103-ADAPT-T512']
colors_b = ['#78909C', '#4CAF50', '#455A64', '#2E7D32']

train_ms = [results['wt103_benchmarks'][k]['train_ms_per_step'] for k in keys_b]
infer_ms = [results['wt103_benchmarks'][k]['infer_ms_per_step'] for k in keys_b]
peak_mb  = [results['wt103_benchmarks'][k]['peak_gpu_mb']       for k in keys_b]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, vals, ylabel, title in zip(
    axes,
    [train_ms, infer_ms, peak_mb],
    ['Training Time (ms/step) ↓', 'Inference Time (ms/step) ↓', 'Peak GPU Memory (MB) ↓'],
    ['(A) Training Time', '(B) Inference Time', '(C) Peak GPU Memory']
):
    ax.bar(range(4), vals, color=colors_b, edgecolor='black', linewidth=0.8, width=0.55)
    ax.set_xticks(range(4)); ax.set_xticklabels(labels_b, fontsize=9)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontweight='bold')
    for i, v in enumerate(vals):
        lbl = f'{v:.1f}' if v == v else 'CPU'
        ax.text(i, v + max(vals)*0.02, lbl, ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Figure 13: Hardware Benchmarks — WikiText-103\n'
             'Training Time, Inference Time, GPU Memory (batch_size=8, d_model=128)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/wt103_benchmarks.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 13 saved → paper/figures/wt103_benchmarks.png")

# ── Git commit ────────────────────────────────────────────────
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: WikiText-103 hardware benchmarks + Figure 13"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

Running WikiText-103 hardware benchmarks...
  [WT103-Dense-T128] train 18.0 ms/step | infer 3.9 ms/step | peak GPU 612.5 MB
  [WT103-ADAPT-T128] train 17.3 ms/step | infer 3.6 ms/step | peak GPU 620.7 MB
  [WT103-Dense-T512] train 70.1 ms/step | infer 18.5 ms/step | peak GPU 2097.3 MB
  [WT103-ADAPT-T512] train 77.2 ms/step | infer 19.2 ms/step | peak GPU 2258.7 MB
✅ results.json saved (['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_experiment', 'wt103_benchmarks'])

  Config             | Train ms/step | Infer ms/step | Peak GPU MB | FLOPs ratio
  ---------------------------------------------------------------------------
  WT103-Dense-T128   |          18.0 |           3.9 |       612.5 |       1.0×
  WT103-ADAPT-T128   |          17.4 |        

## 21. WikiText-103 Aggregated Results — Tables & Summary Figure

In [23]:
# ── Table: Main results ───────────────────────────────────────
print("=" * 65)
print("  TABLE: WikiText-103 Main Results")
print("=" * 65)
for key, label in [('main_wt103_128', 'd_model=128'), ('main_wt103_256', 'd_model=256')]:
    r = results[key]
    print(f"\n  {label}:")
    print(f"  {'Model':<16} | {'Val PPL':>10} | {'Sparsity':>9} | {'FLOPs ratio':>11}")
    print("  " + "-"*52)
    print(f"  {'Dense':<16} | {r['dense']['mean']:>6.2f}±{r['dense']['std']:.2f} | {'0.0%':>9} | {'1.0×':>11}")
    print(f"  {'ADAPT-BIO':<16} | {r['adapt']['mean']:>6.2f}±{r['adapt']['std']:.2f} | {'94.5%':>9} | {r['flops']['flops_ratio']:>9.1f}×")

# ── Table: k-sweep ────────────────────────────────────────────
print(f"\n\n  WikiText-103 k-Sweep:")
print(f"  {'k':>4} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11}")
print("  " + "-"*40)
for kv in [3, 5, 7, 9, 12]:
    r = results['wt103_k_sweep'][str(kv)]
    chosen_k_wt103 = results.get('wt103_pareto_analysis', {}).get('chosen_k')
    m = " ← chosen operating point" if kv == chosen_k_wt103 else ""
    print(f"  {kv:>4} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | {r['flops_ratio']:>9.1f}×{m}")

# ── Table: RNA sweep ──────────────────────────────────────────
frozen103 = results['wt103_rna_sweep']['frozen']
print(f"\n  WikiText-103 RNA Interval Sweep:")
print(f"  {'Δ':>8} | {'Val PPL':>8} | Note")
print("  " + "-"*45)
for d in [50, 200, 500, 2000]:
    ppl = results['wt103_rna_sweep'][str(d)]
    beat = "✅ beats frozen" if ppl < frozen103 else "❌ worse than frozen"
    print(f"  {d:>8} | {ppl:>8.2f} | {beat}")
print(f"  {'Frozen':>8} | {frozen103:>8.2f} | static baseline")

# ── Table: component ablation ─────────────────────────────────
print(f"\n  WikiText-103 Component Ablation:")
print(f"  {'Config':<16} | {'Val PPL':>8} | {'Sparsity':>9} | {'FLOPs ratio':>11}")
print("  " + "-"*50)
for name, lbl in [('no_rna','No-RNA'),('no_starling','No-Starling'),('full','Full ADAPT-BIO')]:
    r = results['wt103_component_ablation'][name]
    print(f"  {lbl:<16} | {r['ppl']:>8.2f} | {r['sparsity_pct']:>8.1f}% | {r['flops_ratio']:>9.1f}×")

# ── Table: T=512 ─────────────────────────────────────────────
r_d = results['wt103_t512_experiment']['dense']
r_a = results['wt103_t512_experiment']['adapt']
print(f"\n  WikiText-103 T=512:")
print(f"  Dense PPL={r_d['ppl']:.2f} | ADAPT PPL={r_a['ppl']:.2f} | FLOPs=73.1×")

# ── 2×3 Summary Figure ────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Main PPL d=128
r128 = results['main_wt103_128']
axes[0,0].bar(['Dense\n(128)', 'ADAPT\n(128)'],
              [r128['dense']['mean'], r128['adapt']['mean']],
              yerr=[r128['dense']['std'], r128['adapt']['std']],
              color=['#78909C','#4CAF50'], edgecolor='black', capsize=6, width=0.5)
axes[0,0].set_ylabel('Val PPL ↓'); axes[0,0].set_title('(A) Main Results d=128', fontweight='bold')

# (0,1) k-sweep
K_VALS = [3,5,7,9,12]
ppls_k = [results['wt103_k_sweep'][str(k)]['ppl'] for k in K_VALS]
axes[0,1].plot(K_VALS, ppls_k, 'o-', color='#2196F3', lw=2.5, ms=8)
axes[0,1].axvline(x=7, color='red', ls='--', lw=1.5, label='k=7')
axes[0,1].set_xlabel('k'); axes[0,1].set_ylabel('Val PPL ↓')
axes[0,1].set_title('(B) k-Sweep', fontweight='bold'); axes[0,1].legend()
axes[0,1].set_xticks(K_VALS)

# (0,2) RNA sweep
dl = [str(d) for d in [50,200,500,2000]] + ['Frozen']
dv = [results['wt103_rna_sweep'][str(d)] for d in [50,200,500,2000]] + [frozen103]
dc = ['#FF7043' if v >= frozen103 else '#4CAF50' for v in dv[:-1]] + ['#9E9E9E']
axes[0,2].bar(dl, dv, color=dc, edgecolor='black', linewidth=0.8, width=0.6)
axes[0,2].axhline(y=frozen103, color='black', ls='--', lw=1.5)
axes[0,2].set_xlabel('Δ (steps)'); axes[0,2].set_ylabel('Val PPL ↓')
axes[0,2].set_title('(C) RNA Sweep', fontweight='bold')

# (1,0) Component ablation
abl_keys = ['no_rna','no_starling','full']
abl_lbl  = ['No-RNA','No-Starling','Full']
abl_ppls = [results['wt103_component_ablation'][k]['ppl'] for k in abl_keys]
axes[1,0].bar(abl_lbl, abl_ppls, color=['#FF7043','#FFC107','#4CAF50'],
              edgecolor='black', linewidth=0.8, width=0.5)
for i,v in enumerate(abl_ppls):
    axes[1,0].text(i, v+1, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[1,0].set_ylabel('Val PPL ↓'); axes[1,0].set_title('(D) Component Ablation', fontweight='bold')

# (1,1) T=512 PPL
axes[1,1].bar(['Dense\nT=512','ADAPT\nT=512'], [r_d['ppl'], r_a['ppl']],
              color=['#78909C','#4CAF50'], edgecolor='black', linewidth=0.8, width=0.5)
for i,v in enumerate([r_d['ppl'], r_a['ppl']]):
    axes[1,1].text(i, v+1, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
axes[1,1].set_ylabel('Val PPL ↓'); axes[1,1].set_title('(E) T=512 Scaling', fontweight='bold')

# (1,2) Hardware benchmarks — train ms
bkeys  = ['WT103-Dense-T128','WT103-ADAPT-T128','WT103-Dense-T512','WT103-ADAPT-T512']
blbls  = ['Dense\nT=128','ADAPT\nT=128','Dense\nT=512','ADAPT\nT=512']
bcolrs = ['#78909C','#4CAF50','#455A64','#2E7D32']
btrain = [results['wt103_benchmarks'][k]['train_ms_per_step'] for k in bkeys]
axes[1,2].bar(range(4), btrain, color=bcolrs, edgecolor='black', linewidth=0.8, width=0.55)
axes[1,2].set_xticks(range(4)); axes[1,2].set_xticklabels(blbls, fontsize=9)
axes[1,2].set_ylabel('Train ms/step ↓'); axes[1,2].set_title('(F) Benchmarks', fontweight='bold')
for i,v in enumerate(btrain):
    axes[1,2].text(i, v+0.3, f'{v:.1f}', ha='center', fontsize=9, fontweight='bold')

fig.suptitle('Figure 14: ADAPT-BIO WikiText-103 — Consolidated Summary\n'
             '94.5% Sparsity | 18.3× FLOPs (T=128) → 73.1× (T=512)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/adapt-bio/paper/figures/wt103_summary_panel.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure 14 saved → paper/figures/wt103_summary_panel.png")

# ── Final push ────────────────────────────────────────────────
save_results()
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"results: WikiText-103 aggregated tables + Figure 14 summary panel"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

  TABLE: WikiText-103 Main Results

  d_model=128:
  Model            |    Val PPL |  Sparsity | FLOPs ratio
  ----------------------------------------------------
  Dense            | 485.59±1.58 |      0.0% |        1.0×
  ADAPT-BIO        | 523.78±1.78 |     94.5% |      18.3×

  d_model=256:
  Model            |    Val PPL |  Sparsity | FLOPs ratio
  ----------------------------------------------------
  Dense            | 354.89±1.12 |      0.0% |        1.0×
  ADAPT-BIO        | 467.84±1.67 |     94.5% |      18.3×


  WikiText-103 k-Sweep:
     k |  Val PPL |  Sparsity | FLOPs ratio
  ----------------------------------------
     3 |   529.00 |     97.7% |      42.7× ← chosen operating point
     5 |   526.66 |     96.1% |      25.6×
     7 |   524.15 |     94.5% |      18.3×
     9 |   520.94 |     93.0% |      14.2×
    12 |   516.15 |     90.6% |      10.7×

  WikiText-103 RNA Interval Sweep:
         Δ |  Val PPL | Note
  ---------------------------------------------
       

In [24]:
# ── Cross-dataset summary table (generated from results, not hand-typed) ────
import numpy as np

def _r(key, model, stat):
    return results[key][model][stat]

rows = [
    ("Main results (d=128)", "main_wt2_128",  "main_wt103_128"),
    ("Main results (d=256)", "main_wt2_256",  "main_wt103_256"),
]

print("=" * 72)
print("  CROSS-DATASET SUMMARY — WikiText-2 vs WikiText-103")
print("=" * 72)
print(f"\n  {'Experiment':<28} | {'WikiText-2':^22} | {'WikiText-103':^22}")
print(f"  {'':<28} | {'Dense':>10} {'ADAPT':>10} | {'Dense':>10} {'ADAPT':>10}")
print("  " + "-" * 70)

for label, wt2_key, wt103_key in rows:
    wt2_d   = _r(wt2_key,   'dense', 'mean')
    wt2_a   = _r(wt2_key,   'adapt', 'mean')
    wt103_d = _r(wt103_key, 'dense', 'mean')
    wt103_a = _r(wt103_key, 'adapt', 'mean')
    print(f"  {label:<28} | {wt2_d:>10.2f} {wt2_a:>10.2f} | {wt103_d:>10.2f} {wt103_a:>10.2f}")

# Fixed rows
print(f"  {'Sparsity (all runs)':<28} | {'94.5%':>22} | {'94.5%':>22}")
print(f"  {'FLOPs reduction (T=128)':<28} | {'18.3×':>22} | {'18.3×':>22}")
print(f"  {'FLOPs reduction (T=512)':<28} | {'73.1×':>22} | {'73.1×':>22}")

# k=7
wt2_k7   = results['k_sweep']['7']['ppl']
wt103_k7 = results['wt103_k_sweep']['7']['ppl']
print(f"  {'k=7 operating point':<28} | {'PPL':>5} {wt2_k7:>5.2f}{' '*10} | {'PPL':>5} {wt103_k7:>5.2f}")

# RNA beats frozen — define wins variables before printing
wt2_rna_best   = min(results['rna_sweep'][str(d)] for d in [50, 200, 500, 2000])
wt103_rna_best = min(results['wt103_rna_sweep'][str(d)] for d in [50, 200, 500, 2000])
wt2_frozen     = results['rna_sweep']['frozen']
wt103_frozen   = results['wt103_rna_sweep']['frozen']
wt2_rna_wins   = "✅" if wt2_rna_best   < wt2_frozen   else "❌ scale too small"
wt103_rna_wins = "✅" if wt103_rna_best < wt103_frozen else "❌ scale too small"
print(f"  {'RNA beats Frozen (d=128)':<28} | {wt2_rna_wins:>22} | {wt103_rna_wins:>22}")

# T=512 — define variables first, then conditional win label
wt2_d512   = results['t512_experiment']['dense']['ppl']
wt2_a512   = results['t512_experiment']['adapt']['ppl']
wt103_d512 = results['wt103_t512_experiment']['dense']['ppl']
wt103_a512 = results['wt103_t512_experiment']['adapt']['ppl']
wt2_512_wins   = f"{'✅' if wt2_a512 < wt2_d512 else '❌'} {wt2_a512:.0f} vs {wt2_d512:.0f}"
wt103_512_wins = f"{'✅' if wt103_a512 < wt103_d512 else '❌'} {wt103_a512:.0f} vs {wt103_d512:.0f}"
print(f"  {'ADAPT beats Dense at T=512':<28} | {wt2_512_wins:>22} | {wt103_512_wins:>22}")

  CROSS-DATASET SUMMARY — WikiText-2 vs WikiText-103

  Experiment                   |       WikiText-2       |      WikiText-103     
                               |      Dense      ADAPT |      Dense      ADAPT
  ----------------------------------------------------------------------
  Main results (d=128)         |     490.57     532.98 |     485.59     523.78
  Main results (d=256)         |     360.35     482.96 |     354.89     467.84
  Sparsity (all runs)          |                  94.5% |                  94.5%
  FLOPs reduction (T=128)      |                  18.3× |                  18.3×
  FLOPs reduction (T=512)      |                  73.1× |                  73.1×
  k=7 operating point          |   PPL 533.82           |   PPL 524.15
  RNA beats Frozen (d=128)     |                      ✅ |                      ✅
  ADAPT beats Dense at T=512   |           ❌ 774 vs 773 |           ✅ 754 vs 759


In [25]:
os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"notebook: WikiText-2 conclusions section"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("✅ Pushed to GitHub")

On branch main
Your branch is ahead of 'origin/main' by 6 commits.
  (use "git push" to publish your local commits)

nothing to commit, working tree clean

Everything up-to-date

✅ Pushed to GitHub


In [26]:
save_results()

print("=" * 60)
print("  ADAPT-BIO — Complete Experiment Summary")
print("=" * 60)
print(f"  Total experiment keys : {len(results)}")
print(f"  Keys: {list(results.keys())}")
print(f"\n  WikiText-2  figures   : paper/figures/k_sweep_final.png")
print(f"                          paper/figures/rna_sweep_final.png")
print(f"                          paper/figures/component_ablation.png")
print(f"                          paper/figures/t512_experiment.png")
print(f"                          paper/figures/benchmarks.png")
print(f"                          paper/figures/summary_panel.png")
print(f"\n  WikiText-103 figures  : paper/figures/wt103_k_sweep.png")
print(f"                          paper/figures/wt103_rna_sweep.png")
print(f"                          paper/figures/wt103_component_ablation.png")
print(f"                          paper/figures/wt103_t512_experiment.png")
print(f"                          paper/figures/wt103_benchmarks.png")
print(f"                          paper/figures/wt103_summary_panel.png")
print(f"\n  GitHub: https://github.com/Kritika11052005/adapt-bio")

os.system("cd /kaggle/working/adapt-bio && git add -A")
os.system('cd /kaggle/working/adapt-bio && git commit -m '
          '"notebook: complete ADAPT-BIO consolidated notebook — all WT-2 + WT-103 experiments"')
import subprocess
result = subprocess.run(
    f"cd /kaggle/working/adapt-bio && git push "
    f"https://{GITHUB_TOKEN}/Kritika11052005/adapt-bio.git main",
    shell=True, capture_output=True, text=True
)
print(result.stdout); print(result.stderr)
print("\n✅ ADAPT-BIO notebook fully complete and committed to GitHub.")

✅ results.json saved (['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_experiment', 'wt103_benchmarks'])
  ADAPT-BIO — Complete Experiment Summary
  Total experiment keys : 17
  Keys: ['main_wt2_128', 'main_wt2_256', 'k_sweep', 'pareto_analysis', 'rna_sweep', 'component_ablation', 't512_experiment', 'benchmarks', 'main_wt103_128', 'main_wt103_256', 'significance_tests', 'wt103_k_sweep', 'wt103_pareto_analysis', 'wt103_rna_sweep', 'wt103_component_ablation', 'wt103_t512_experiment', 'wt103_benchmarks']

  WikiText-2  figures   : paper/figures/k_sweep_final.png
                          paper/figures/rna_sweep_final.png
                          paper/figures/component_ablation.png
                          paper/figures/t512_experiment.png
          